In [16]:
import pandas as pd 
import glob
import os

In [4]:

def read_fact_data_by_month(month_str: str):
    # Giả sử month_str có dạng "202412"
    # Tách 4 ký tự đầu làm năm, phần còn lại làm tháng
    year = int(month_str[:4])
    mon = int(month_str[4:])
    
    # Start date (YYYYMM01)
    start = int(f"{year}{mon:02d}01")
    
    # Last day of month
    last_day = calendar.monthrange(year, mon)[1]
    stop = int(f"{year}{mon:02d}{last_day}")
    
    # Read parquet với filter
    # Lưu ý: Đảm bảo đường dẫn thư mục chính xác
    df = pd.read_parquet(
        "../data/data_hitech_day/",
        filters=[
            ("partition", ">=", start),
            ("partition", "<=", stop)
        ],
        engine="pyarrow"
    )
    
    # Groupby và gom nhóm dữ liệu
    data = df.groupby("cus_id").agg({
        "tong_don": "sum",
        "tong_tien": "sum",
        "don_ptc": "sum",
        "thu_ho": "sum",
        "don_hoan": "sum"
    }).reset_index()
    
    # Thêm cột tháng theo định dạng đầu vào ban đầu
    data['month'] = month_str
    
    return data

# Cách gọi hàm:
# df_result = read_data_by_month("202412")

In [12]:
import pyarrow.dataset as ds

# Thay vì dùng glob, bạn trỏ thẳng vào thư mục gốc hoặc danh sách file
# Lưu ý: dataset cần cấu trúc thư mục chuẩn partition=value
dataset = ds.dataset(file_list, format="parquet", partitioning="hive")

# Chuyển thành Pandas DataFrame (nó sẽ tự có sẵn cột 'partition')
df = dataset.to_table().to_pandas()

# Nếu muốn đổi tên cột từ 'partition' thành 'order_date'
df = df.rename(columns={'partition': 'order_date'})
df.head()

,cus_id,ten_phuongxa_hoatdong_chinh,ten_quanhuyen_hoatdong_chinh,ma_tinh_hoatdong_chinh,tongdon_diaban_chinh,tong_tien,min_tongtien,max_tongtien,tong_don,avg_tongtien,...,so_khieunai_nhieunhat,ngay_hoptac,ngay_sinh,ten_dangky,nganh_hang,loai_khachhang,thuho_tongdon,tongdon_cod,don_ptc_cod,order_date
0,13013968,PHƯỜNG BÌNH HƯNG HÒA,QUẬN BÌNH TÂN,HCM,1,79503.0,79503.0,79503.0,1,79503.00,...,NaN,03/07/2023,None,CÙ THANH LỘC,Khác,Cá nhân,0.0,0,0.0,20260314
1,6078374,THỊ TRẤN CẦU GIÁT,HUYỆN QUỲNH LƯU,NAN,1,13000.0,13000.0,13000.0,1,13000.00,...,NaN,20/01/2019,517363200000,TRẦN THỊ VIỆN,Khác,Hộ kinh doanh cá thể,0.0,0,0.0,20260314
2,12473520,XÃ TÂN THÀNH BÌNH,HUYỆN MỎ CÀY BẮC,BTE,1,21000.0,21000.0,21000.0,1,21000.00,...,NaN,12/01/2023,630115200000,Nhi,Khác,Hộ kinh doanh cá thể,1240000.0,1,0.0,20260314
3,15681153,THỊ TRẤN LIÊN HƯƠNG,HUYỆN TUY PHONG,BTN,1,30600.0,30600.0,30600.0,1,30600.00,...,NaN,21/01/2025,None,Qua Thụy Bạch,Khác,Cá nhân,0.0,0,NaN,20260314
4,17757633,THỊ TRẤN CỔ LỄ,HUYỆN TRỰC NINH,NDH,4,115501.0,25000.0,32000.0,4,28875.25,...,NaN,26/01/2026,None,Trần việt hà,None,Cá nhân,3843000.0,4,2.0,20260314


In [11]:
import pyarrow.parquet as pq
month_str = "202603"
file_prefix = "20260410"
base_path = "../data/data_hitech_day/"
search_pattern = f"{base_path}partition={month_str}*/{file_prefix}*"
file_list = glob.glob(search_pattern, recursive=True)
for file_path in file_list:
    # Chia chuỗi tại 'partition=', lấy phần bên phải, sau đó lấy phần trước dấu gạch chéo đầu tiên
    partition_value = file_path.split("partition=")[1].split("/")[0]
    
    print(f"File: {file_path}")
    print(f"Giá trị Partition: {partition_value}")

File: ../data/data_hitech_day/partition=20260314/20260410_225509_36924_xxcuu_e259829b-f134-4eae-b97c-18d0c96cca5e
Giá trị Partition: 20260314
File: ../data/data_hitech_day/partition=20260320/20260410_225823_36968_xxcuu_c4f77f88-3c85-4570-bf01-380a59747dd5
Giá trị Partition: 20260320
File: ../data/data_hitech_day/partition=20260329/20260410_230949_37517_xxcuu_ca50b5ce-3de3-4f1b-ab5f-be1d19ddd7b1
Giá trị Partition: 20260329
File: ../data/data_hitech_day/partition=20260310/20260410_225322_36900_xxcuu_e580d614-8f49-49ec-a78a-48f6ddc8728a
Giá trị Partition: 20260310
File: ../data/data_hitech_day/partition=20260330/20260410_231020_37527_xxcuu_b1649806-4e04-4493-9de8-028c501ce6b7
Giá trị Partition: 20260330
File: ../data/data_hitech_day/partition=20260302/20260410_224727_36832_xxcuu_3387e888-3250-4454-a076-f07edbbaffa3
Giá trị Partition: 20260302
File: ../data/data_hitech_day/partition=20260317/20260410_225652_36951_xxcuu_77fdbd06-c5d5-4fc9-a4cf-1fa39767737d
Giá trị Partition: 20260317
File: 

In [63]:
import pandas as pd
import glob
import os

def read_fact_data_with_file_filter(month_str: str, file_prefix: str):
    year = int(month_str[:4])
    mon = int(month_str[4:])
    
    # Đường dẫn gốc
    base_path = "../data/data_hitech_day/"
    
    # 1. Sử dụng glob để tìm tất cả các file thỏa mãn:
    # - Nằm trong các folder partition thuộc năm/tháng đó
    # - Tên file bắt đầu bằng file_prefix (20260410)
    # Cấu trúc: base_path/partition=YYYYMMDD/prefix*
    search_pattern = f"{base_path}partition={month_str}*/{file_prefix}*"
    
    # recursive=True giúp tìm kiếm sâu hơn nếu có folder con
    file_list = glob.glob(search_pattern, recursive=True)
    
    if not file_list:
        print(f"Không tìm thấy file nào khớp với prefix {file_prefix} trong tháng {month_str}")
        return None

    # 2. Đọc tất cả các file tìm thấy (nếu có nhiều file cùng prefix đó)
    # Pandas read_parquet có thể nhận vào 1 list các đường dẫn file
    df = pd.read_parquet(file_list, engine="pyarrow")
    
    # 3. Thực hiện Groupby như cũ
    data = df.groupby("cus_id").agg({
            "tong_don": "sum",
            "tong_tien": "sum",
            "tongdon_cod": "sum",
            "thuho_tongdon": "sum",
            "don_ptc": "sum",
            "don_ptc_cod": "sum",
            "thu_ho": "sum"
        }).reset_index()
    
    data['month'] = month_str
    return data

# Gọi hàm
# Nó sẽ chỉ đọc những file bắt đầu bằng "20260410" 
# nằm trong các thư mục partition bắt đầu bằng "202412"
# df_result = read_fact_data_with_file_filter("202603", "20260410")
# df_result.head()

In [2]:
def read_dim_data_by_month(month_str: str):
    # Giả sử month_str có dạng "202412"
    # Tách 4 ký tự đầu làm năm, phần còn lại làm tháng
    year = int(month_str[:4])
    mon = int(month_str[4:])
    
    # Start date (YYYYMM01)
    start = int(f"{year}{mon:02d}01")
    
    # Last day of month
    last_day = calendar.monthrange(year, mon)[1]
    stop = int(f"{year}{mon:02d}{last_day}")
    
    # Read parquet với filter
    # Lưu ý: Đảm bảo đường dẫn thư mục chính xác
    df = pd.read_parquet(
        "../data/data_hitech_day/",
        filters=[
            ("partition", ">=", start),
            ("partition", "<=", stop)
        ],
        engine="pyarrow", columns=['cus_id','ngay_hoptac', 'ngay_sinh', 'ten_dangky', 'nganh_hang',
       'loai_khachhang']
    )
    return df

In [17]:
def read_dim_data_with_file_filter(month_str: str, file_prefix: str):
    year = int(month_str[:4])
    mon = int(month_str[4:])
    
    # Đường dẫn gốc
    base_path = "../data/data_hitech_day/"
    
    # 1. Sử dụng glob để tìm tất cả các file thỏa mãn:
    # - Nằm trong các folder partition thuộc năm/tháng đó
    # - Tên file bắt đầu bằng file_prefix (20260410)
    # Cấu trúc: base_path/partition=YYYYMMDD/prefix*
    search_pattern = f"{base_path}partition={month_str}*/{file_prefix}*"
    
    # recursive=True giúp tìm kiếm sâu hơn nếu có folder con
    file_list = glob.glob(search_pattern, recursive=True)
    
    if not file_list:
        print(f"Không tìm thấy file nào khớp với prefix {file_prefix} trong tháng {month_str}")
        return None

    # 2. Đọc tất cả các file tìm thấy (nếu có nhiều file cùng prefix đó)
    # Pandas read_parquet có thể nhận vào 1 list các đường dẫn file
    dataset = ds.dataset(file_list, format="parquet", partitioning="hive")

    columns_to_read = ['cus_id', 'ngay_hoptac', 'partition']
    # Chuyển thành Pandas DataFrame (nó sẽ tự có sẵn cột 'partition')
    df = dataset.to_table(columns=columns_to_read).to_pandas()
    
    # Nếu muốn đổi tên cột từ 'partition' thành 'order_date'
    df = df.rename(columns={'partition': 'order_date'})
    
    return df

In [22]:
all_folders = sorted([f for f in os.listdir(r'../data/data_hitech_month')])
file_prefix = "20260410"
# 2. Lấy 12 folder cuối cùng
last_12_folders = all_folders[-12:]
for folder in last_12_folders:
    month = folder.split('=')[1]
    dim = read_dim_data_with_file_filter(month,file_prefix)
    dim.to_csv(f'../monthly_grouped_data/dim_data/{month}.csv', index = False)

In [65]:
all_folders = sorted([f for f in os.listdir(r'../data/data_hitech_month')])
file_prefix = "20260410"
# 2. Lấy 12 folder cuối cùng
last_12_folders = all_folders[-12:]
for folder in last_12_folders:
    month = folder.split('=')[1]
    fact = read_fact_data_with_file_filter(month,file_prefix)
    fact.to_csv(f'../monthly_grouped_data/fact_data/{month}.csv', index = False)

In [17]:
import glob 
frames = []
for file in glob.glob(r'../monthly_grouped_data/*.csv'):
    df = pd.read_csv(file)
    frames.append(df)
all_data = pd.concat(frames)
all_data.to_csv(r'../monthly_grouped_data/fact_data/fact_data.csv', index = False)

In [6]:
import os
import calendar
for folder in os.listdir(r'../data/data_hitech_month'):
    month = folder.split('=')[1]
    df = read_dim_data_by_month(month)
    df.to_csv(f'../monthly_grouped_data/dim_data/{month}.csv', index = False)

#### Dim data

In [23]:
import glob 
frames = []
for file in glob.glob(r'../monthly_grouped_data/dim_data/*.csv'):
    df = pd.read_csv(file)
    frames.append(df)
dim_data = pd.concat(frames)
dim_data.to_csv(r'../monthly_grouped_data/temp/12month_dim_data.csv', index = False)

In [24]:
dim_data.head()

,cus_id,ngay_hoptac,order_date
0,15521383,19/12/2024,20250414
1,3983710,03/01/2019,20250414
2,15888055,13/03/2025,20250414
3,15409953,27/11/2024,20250414
4,5947192,18/01/2019,20250414


In [26]:
df1 = dim_data[['cus_id','ngay_hoptac']].drop_duplicates().reset_index(drop = True)
df1.head()

,cus_id,ngay_hoptac
0,15521383,19/12/2024
1,3983710,03/01/2019
2,15888055,13/03/2025
3,15409953,27/11/2024
4,5947192,18/01/2019


In [29]:
import pandas as pd
import numpy as np
import datetime

# Lấy thời điểm hiện tại một lần để đồng nhất dữ liệu
now = pd.Timestamp.now()

# 1. Chuyển đổi định dạng ngày
df1['ngay_hoptac'] = pd.to_datetime(df1['ngay_hoptac'], format='%d/%m/%Y')

# 2. Tính số tháng sử dụng (Sửa dấu trừ thành dấu cộng ở phần tháng)
df1['thoigian_sd'] = (now.year - df1['ngay_hoptac'].dt.year) * 12 + (now.month - df1['ngay_hoptac'].dt.month)

# 3. Gán flag (Sửa dim_data thành df1)
df1['tgsd_flag'] = np.where(df1['thoigian_sd'] >= 6, 1, 0)

df1.head()

,cus_id,ngay_hoptac,thoigian_sd,tgsd_flag
0,15521383,2024-12-19,16.0,1
1,3983710,2019-01-03,87.0,1
2,15888055,2025-03-13,13.0,1
3,15409953,2024-11-27,17.0,1
4,5947192,2019-01-18,87.0,1


In [30]:
df1 = df1[['cus_id','tgsd_flag']]

In [31]:
df2 = dim_data.groupby("cus_id").agg({
    "order_date": "min"
}).reset_index()
df2

,cus_id,order_date
0,1,20250401
1,7,20250401
2,20,20250401
3,28,20260107
4,34,20250426
...,...,...
2522617,18158971,20260331
2522618,18159038,20260331
2522619,18159060,20260331
2522620,18159087,20260331


In [33]:
df2['order_date'] = pd.to_datetime(df2['order_date'], format='%Y%m%d')

# 2. Tính số tháng sử dụng (Sửa dấu trừ thành dấu cộng ở phần tháng)
df2['thoigian_sd'] = (now.year - df2['order_date'].dt.year) * 12 + (now.month - df2['order_date'].dt.month)

# 3. Gán flag (Sửa dim_data thành df1)
df2['first_order_12moth_flag'] = np.where(df2['thoigian_sd'] >= 6, 1, 0)
df2.head()

,cus_id,order_date,thoigian_sd,first_order_12moth_flag
0,1,2025-04-01,12,1
1,7,2025-04-01,12,1
2,20,2025-04-01,12,1
3,28,2026-01-07,3,0
4,34,2025-04-26,12,1


In [34]:
df2 = df2[['cus_id','first_order_12moth_flag']]
df2.head()

,cus_id,first_order_12moth_flag
0,1,1
1,7,1
2,20,1
3,28,0
4,34,1


In [38]:
dim_data_12month = pd.merge(df1, df2, how = 'inner', on = ['cus_id'])
dim_data_12month

,cus_id,tgsd_flag,first_order_12moth_flag
0,15521383,1,1
1,3983710,1,1
2,15888055,1,1
3,15409953,1,1
4,5947192,1,1
...,...,...,...
2522617,17691754,0,0
2522618,18128299,0,0
2522619,13202415,1,0
2522620,1969526,1,0


In [39]:
dim_data_12month.to_csv(r'../monthly_grouped_data/temp/12month_dim_data_aggre.csv', index = False)

In [66]:
frames = []
for file in glob.glob(r'../monthly_grouped_data/fact_data/*.csv'):
    df = pd.read_csv(file)
    frames.append(df)
fact_data = pd.concat(frames)
fact_data = fact_data.drop_duplicates().reset_index()
fact_data.to_csv(r'../monthly_grouped_data/temp/12month_fact_data.csv', index = False)

In [11]:
fact_data.head()

,index,cus_id,tong_don,tongdon_cod,thuho_tongdon,don_ptc,don_ptc_cod,thu_ho,month
0,0,1,2,2,1440000.0,0.0,0.0,0.0,202504
1,1,10000004,4,0,0.0,0.0,0.0,0.0,202504
2,2,10000027,1,0,0.0,0.0,0.0,0.0,202504
3,3,10000085,49,38,34196000.0,41.0,32.0,30585000.0,202504
4,4,10000108,2,2,13150000.0,0.0,0.0,0.0,202504


In [161]:
### đối soát outlier vs Mai vtp
fact_data_temp1 = pd.read_csv(r'../monthly_grouped_data/temp/12month_fact_data.csv')
fact_data_year_temp1 = fact_data_temp1.groupby(["cus_id"]).agg({
    "tong_don": "sum",
    "tong_tien": "sum",
    "tongdon_cod": "sum",
    "thuho_tongdon": "sum",
    "don_ptc": "sum",
    "don_ptc_cod": "sum",
    "thu_ho": "sum"
}).reset_index()

In [162]:
fact_data_year_temp1['tile_ptc'] = fact_data_year_temp1['don_ptc'] / fact_data_year_temp1['tong_don']

In [167]:
fact_data_year_temp1[(fact_data_year_temp1['tile_ptc'] < 0.01) & (fact_data_year_temp1['don_ptc'] != 0)]

,cus_id,tong_don,tong_tien,tongdon_cod,thuho_tongdon,don_ptc,don_ptc_cod,thu_ho,tile_ptc
619,6523,127,3718162.0,1,6.200000e+06,1.0,0.0,0.0,0.007874
3503,46411,3270,78800729.0,1615,1.139587e+09,5.0,0.0,0.0,0.001529
6626,81618,214,7046233.0,0,0.000000e+00,1.0,0.0,0.0,0.004673
8299,99986,1773,107323353.0,0,0.000000e+00,1.0,0.0,0.0,0.000564
10601,146910,119,5836660.0,0,0.000000e+00,1.0,0.0,0.0,0.008403
...,...,...,...,...,...,...,...,...,...
2487309,18025755,344,12307252.0,0,0.000000e+00,1.0,0.0,0.0,0.002907
2510603,18109013,3095,30965700.0,0,0.000000e+00,4.0,0.0,0.0,0.001292
2515046,18123104,128,4793262.0,128,1.447900e+08,1.0,1.0,1150000.0,0.007812
2516114,18126490,632,6320000.0,0,0.000000e+00,6.0,0.0,0.0,0.009494


### EDA all data

In [1]:
import pandas as pd
import datetime

#### 1 tổng port

In [2]:
dim_data = pd.read_csv(r'../monthly_grouped_data/temp/12month_dim_data.csv', usecols = ['cus_id','ngay_hoptac'])
dim_data = dim_data.drop_duplicates().reset_index(drop = True)
dim_data['ngay_hoptac'] = pd.to_datetime(dim_data['ngay_hoptac'], format='%d/%m/%Y')
now = pd.Timestamp.now()
# 2. Tính số tháng sử dụng (Sửa dấu trừ thành dấu cộng ở phần tháng)
dim_data['thoigian_sd'] = (now.year - dim_data['ngay_hoptac'].dt.year) * 12 + (now.month - dim_data['ngay_hoptac'].dt.month)
dim_data.head()

,cus_id,ngay_hoptac,thoigian_sd
0,15521383,2024-12-19,16.0
1,3983710,2019-01-03,87.0
2,15888055,2025-03-13,13.0
3,15409953,2024-11-27,17.0
4,5947192,2019-01-18,87.0


#### 2. TGSD >= 4

In [97]:
dim_data = dim_data[dim_data['thoigian_sd'] >= 4]

In [98]:
rule_dataset = dim_data.copy()
rule_dataset.shape

(2349929, 3)

#### 3. 4 tháng gần nhất k có tháng nào có 0 đơn ptc

In [102]:
fact_data_temp = pd.read_csv(r'../monthly_grouped_data/temp/12month_fact_data.csv')
fact_data_temp = pd.merge(fact_data_temp,rule_dataset, how = 'inner', on = 'cus_id')
fact_data_temp.head()

,index,cus_id,tong_don,tong_tien,tongdon_cod,thuho_tongdon,don_ptc,don_ptc_cod,thu_ho,month,ngay_hoptac,thoigian_sd
0,0,1,2,66000.0,2,1440000.0,0.0,0.0,0.0,202504,2017-09-28,103.0
1,1,10000004,4,111999.0,0,0.0,0.0,0.0,0.0,202504,2021-08-23,56.0
2,2,10000027,1,217500.0,0,0.0,0.0,0.0,0.0,202504,2021-08-23,56.0
3,3,10000085,49,1547568.0,38,34196000.0,41.0,32.0,30585000.0,202504,2021-08-23,56.0
4,4,10000108,2,97103.0,2,13150000.0,0.0,0.0,0.0,202504,2021-08-23,56.0


In [103]:
fact_data_temp.shape

(8832810, 12)

In [104]:
# Lấy đúng 6 tháng gần nhất
last_4_months = sorted(fact_data_temp['month'].unique())[-4:]

# Lọc data
check_data = fact_data_temp[fact_data_temp['month'].isin(last_4_months)].copy()

# Pivot
pivot_df = check_data.pivot_table(
    index='cus_id', 
    columns='month', 
    values='don_ptc', 
    aggfunc='sum'
)

# Đảm bảo đủ 4 tháng
pivot_df = pivot_df.reindex(columns=last_4_months).fillna(0)

# Khách có ít nhất 1 tháng không có đơn
cus_co_thang_trong = pivot_df[(pivot_df == 0).any(axis=1)]

# List ID
list_cus_trong = cus_co_thang_trong.index.tolist()

cus_pass = pivot_df[(pivot_df != 0).all(axis=1)]
list_cus_pass = cus_pass.index.tolist()

total_cus = pivot_df.shape[0]
num_cus_trong = len(list_cus_trong)
num_cus_full = len(list_cus_pass)

print("Tổng KH:", total_cus)
print("KH có tháng trống:", num_cus_trong)
print("KH đủ 4 tháng:", num_cus_full)

Tổng KH: 1317576
KH có tháng trống: 1190365
KH đủ 4 tháng: 127211


In [106]:
fact_data_temp = fact_data_temp[fact_data_temp['cus_id'].isin(list_cus_pass)]
fact_data_temp = fact_data_temp.drop(columns = ['index'])


,index,cus_id,tong_don,tong_tien,tongdon_cod,thuho_tongdon,don_ptc,don_ptc_cod,thu_ho,month,ngay_hoptac,thoigian_sd
8,8,10000201,34,584005.0,0,0.0,14.0,0.0,0.0,202504,2021-08-23,56.0
23,23,10000725,29,355000.0,0,0.0,13.0,0.0,0.0,202504,2021-08-23,56.0
32,32,10001064,240,9807436.0,176,131930000.0,218.0,164.0,130380000.0,202504,2021-08-23,56.0
34,34,10001144,6,426106.0,0,0.0,0.0,0.0,0.0,202504,2021-08-23,56.0
40,40,10001236,482,15914658.0,410,418589000.0,462.0,392.0,394408000.0,202504,2021-08-23,56.0
...,...,...,...,...,...,...,...,...,...,...,...,...
8832772,782677,9999040,20,400353.0,9,6320000.0,10.0,3.0,700000.0,202603,2021-08-22,56.0
8832777,782682,9999187,8,1028015.0,4,11515000.0,2.0,1.0,6220000.0,202603,2021-08-23,56.0
8832789,782694,9999337,47,4069019.0,7,4530000.0,26.0,6.0,4080000.0,202603,2021-08-23,56.0
8832799,782704,9999644,21,450897.0,15,20242000.0,3.0,2.0,3410000.0,202603,2021-08-23,56.0


In [ ]:
fact_data_year_thu_ho['chi_phi_ptc'] = fact_data_year_thu_ho['don_ptc'] * ( fact_data_year_thu_ho['tong_tien'] / fact_data_year_thu_ho['tong_don'] ) 

#### 2.Vẽ matrix

##### Rule1: Có ít nhất 1 đơn ptc

In [109]:
fact_data_year_temp = fact_data_temp.groupby(["cus_id"]).agg({
    "tong_don": "sum",
    "tong_tien": "sum",
    "tongdon_cod": "sum",
    "thuho_tongdon": "sum",
    "don_ptc": "sum",
    "don_ptc_cod": "sum",
    "thu_ho": "sum"
}).reset_index()
fact_data_year = pd.merge(rule_dataset,fact_data_year_temp, how = 'inner', on = 'cus_id')
fact_data_year['chi_phi_ptc'] = fact_data_year['don_ptc'] * ( fact_data_year['tong_tien'] / fact_data_year['tong_don'] ) 
fact_data_year.head()                 

,cus_id,ngay_hoptac,thoigian_sd,tong_don,tong_tien,tongdon_cod,thuho_tongdon,don_ptc,don_ptc_cod,thu_ho,chi_phi_ptc
0,3815832,2019-01-01,87.0,323,11286708.0,50,167240460.0,160.0,33.0,96525628.0,5.590939e+06
1,13552722,2023-11-14,29.0,197,23608628.0,4,7400000.0,82.0,1.0,3420000.0,9.826942e+06
2,13845857,2024-01-26,27.0,573,7193095.0,0,0.0,349.0,0.0,0.0,4.381135e+06
3,7976779,2020-06-18,70.0,764,42656147.0,86,24619920.0,600.0,73.0,10859982.0,3.349959e+07
4,14988133,2024-09-03,19.0,203,6339667.0,98,32823500.0,65.0,28.0,15283000.0,2.029943e+06


In [110]:
fact_data_year.shape

(127211, 11)

##### tách 2 nhóm để tính GMV

##### Nhóm 1. Có thu hộ

In [111]:
fact_data_year_thu_ho = fact_data_year[((fact_data_year['thu_ho'] != 0) | (fact_data_year['thuho_tongdon'] != 0))].reset_index(drop = True)
fact_data_year_thu_ho

,cus_id,ngay_hoptac,thoigian_sd,tong_don,tong_tien,tongdon_cod,thuho_tongdon,don_ptc,don_ptc_cod,thu_ho,chi_phi_ptc
0,3815832,2019-01-01,87.0,323,11286708.0,50,1.672405e+08,160.0,33.0,9.652563e+07,5.590939e+06
1,13552722,2023-11-14,29.0,197,23608628.0,4,7.400000e+06,82.0,1.0,3.420000e+06,9.826942e+06
2,7976779,2020-06-18,70.0,764,42656147.0,86,2.461992e+07,600.0,73.0,1.085998e+07,3.349959e+07
3,14988133,2024-09-03,19.0,203,6339667.0,98,3.282350e+07,65.0,28.0,1.528300e+07,2.029943e+06
4,14067926,2024-03-22,25.0,3123,88299650.0,1731,2.532835e+09,2728.0,1497.0,2.113407e+09,7.713143e+07
...,...,...,...,...,...,...,...,...,...,...,...
87496,10185158,2021-09-24,55.0,80,1871217.0,65,1.856000e+07,44.0,35.0,9.405000e+06,1.029169e+06
87497,7104910,2019-11-08,77.0,43,2726078.0,13,5.951700e+07,15.0,3.0,1.603500e+07,9.509574e+05
87498,17495758,2025-12-18,4.0,270,5221779.0,219,8.165001e+07,167.0,135.0,5.082301e+07,3.229767e+06
87499,10669683,2021-11-23,53.0,35,1892695.0,20,2.452510e+07,10.0,5.0,6.801000e+06,5.407700e+05


In [112]:
fact_data_year_thu_ho['rev_per_don_ptc'] =  (fact_data_year_thu_ho['thu_ho'] / fact_data_year_thu_ho['don_ptc_cod']).fillna(0)
fact_data_year_thu_ho['rev_per_tongdon'] =  (fact_data_year_thu_ho['thuho_tongdon'] / fact_data_year_thu_ho['tongdon_cod']).fillna(0)

In [41]:
# tính trung bình 
# fact_data_year_thu_ho['gmv_case1'] = fact_data_year_thu_ho['don_ptc'] * ( ( fact_data_year_thu_ho['rev_per_don_ptc'] + fact_data_year_thu_ho['rev_per_tongdon'] ) / 2)

In [113]:
# tính theo ưu tiên đơn ptc có cod
fact_data_year_thu_ho['gmv_case2'] = (
    fact_data_year_thu_ho['don_ptc'] *
    fact_data_year_thu_ho['rev_per_don_ptc'].where(
        fact_data_year_thu_ho['rev_per_don_ptc'] != 0,
        fact_data_year_thu_ho['rev_per_tongdon']
    )
)

In [114]:
# fact_data_year_thu_ho['%chi_phi_ptc_case1'] = fact_data_year_thu_ho['chi_phi_ptc'] / fact_data_year_thu_ho['gmv_case1']
fact_data_year_thu_ho['%chi_phi_ptc_case2'] = fact_data_year_thu_ho['chi_phi_ptc'] / fact_data_year_thu_ho['gmv_case2']

In [115]:
fact_data_year_thu_ho

,cus_id,ngay_hoptac,thoigian_sd,tong_don,tong_tien,tongdon_cod,thuho_tongdon,don_ptc,don_ptc_cod,thu_ho,chi_phi_ptc,rev_per_don_ptc,rev_per_tongdon,gmv_case2,%chi_phi_ptc_case2
0,3815832,2019-01-01,87.0,323,11286708.0,50,1.672405e+08,160.0,33.0,9.652563e+07,5.590939e+06,2.925019e+06,3.344809e+06,4.680030e+08,0.011946
1,13552722,2023-11-14,29.0,197,23608628.0,4,7.400000e+06,82.0,1.0,3.420000e+06,9.826942e+06,3.420000e+06,1.850000e+06,2.804400e+08,0.035041
2,7976779,2020-06-18,70.0,764,42656147.0,86,2.461992e+07,600.0,73.0,1.085998e+07,3.349959e+07,1.487669e+05,2.862781e+05,8.926013e+07,0.375303
3,14988133,2024-09-03,19.0,203,6339667.0,98,3.282350e+07,65.0,28.0,1.528300e+07,2.029943e+06,5.458214e+05,3.349337e+05,3.547839e+07,0.057216
4,14067926,2024-03-22,25.0,3123,88299650.0,1731,2.532835e+09,2728.0,1497.0,2.113407e+09,7.713143e+07,1.411762e+06,1.463221e+06,3.851286e+09,0.020027
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
87496,10185158,2021-09-24,55.0,80,1871217.0,65,1.856000e+07,44.0,35.0,9.405000e+06,1.029169e+06,2.687143e+05,2.855385e+05,1.182343e+07,0.087045
87497,7104910,2019-11-08,77.0,43,2726078.0,13,5.951700e+07,15.0,3.0,1.603500e+07,9.509574e+05,5.345000e+06,4.578231e+06,8.017500e+07,0.011861
87498,17495758,2025-12-18,4.0,270,5221779.0,219,8.165001e+07,167.0,135.0,5.082301e+07,3.229767e+06,3.764667e+05,3.728311e+05,6.286994e+07,0.051372
87499,10669683,2021-11-23,53.0,35,1892695.0,20,2.452510e+07,10.0,5.0,6.801000e+06,5.407700e+05,1.360200e+06,1.226255e+06,1.360200e+07,0.039757


##### Nhóm 2. Không có thu hộ

In [137]:
fact_data_year_not_thu_ho = fact_data_year[(fact_data_year['thu_ho'] == 0) & (fact_data_year['thuho_tongdon'] == 0)].reset_index(drop = True)
fact_data_year_not_thu_ho

,cus_id,ngay_hoptac,thoigian_sd,tong_don,tong_tien,tongdon_cod,thuho_tongdon,don_ptc,don_ptc_cod,thu_ho,chi_phi_ptc
0,13845857,2024-01-26,27.0,573,7193095.0,0,0.0,349.0,0.0,0.0,4.381135e+06
1,12627701,2023-03-08,37.0,1388,54142752.0,0,0.0,1188.0,0.0,0.0,4.634120e+07
2,7551727,2020-03-13,73.0,440,21306919.0,0,0.0,239.0,0.0,0.0,1.157353e+07
3,9216277,2021-05-07,59.0,283,4337732.0,0,0.0,148.0,0.0,0.0,2.268496e+06
4,12105262,2022-09-27,43.0,911,10579194.0,0,0.0,596.0,0.0,0.0,6.921185e+06
...,...,...,...,...,...,...,...,...,...,...,...
39705,13412500,2023-10-11,30.0,70,2327877.0,0,0.0,21.0,0.0,0.0,6.983631e+05
39706,17486356,2025-12-18,4.0,53,1288459.0,0,0.0,26.0,0.0,0.0,6.320742e+05
39707,13992059,2024-03-04,25.0,259,6629961.0,0,0.0,197.0,0.0,0.0,5.042866e+06
39708,17496297,2025-12-18,4.0,57,9167048.0,0,0.0,20.0,0.0,0.0,3.216508e+06


In [139]:
# tinh med và mean của gmv ở tệp có thu hộ sau khi đã loại đi gmv >= xxx
fact_data_year_not_thu_ho['gmv_med'] = fact_data_year_not_thu_ho['chi_phi_ptc'] / 0.044734
fact_data_year_not_thu_ho['gmv_mean'] = fact_data_year_not_thu_ho['chi_phi_ptc'] / 0.065371
fact_data_year_not_thu_ho

,cus_id,ngay_hoptac,thoigian_sd,tong_don,tong_tien,tongdon_cod,thuho_tongdon,don_ptc,don_ptc_cod,thu_ho,chi_phi_ptc,gmv_med,gmv_mean
0,13845857,2024-01-26,27.0,573,7193095.0,0,0.0,349.0,0.0,0.0,4.381135e+06,9.793747e+07,6.701954e+07
1,12627701,2023-03-08,37.0,1388,54142752.0,0,0.0,1188.0,0.0,0.0,4.634120e+07,1.035928e+09,7.088954e+08
2,7551727,2020-03-13,73.0,440,21306919.0,0,0.0,239.0,0.0,0.0,1.157353e+07,2.587189e+08,1.770438e+08
3,9216277,2021-05-07,59.0,283,4337732.0,0,0.0,148.0,0.0,0.0,2.268496e+06,5.071078e+07,3.470187e+07
4,12105262,2022-09-27,43.0,911,10579194.0,0,0.0,596.0,0.0,0.0,6.921185e+06,1.547187e+08,1.058755e+08
...,...,...,...,...,...,...,...,...,...,...,...,...,...
39705,13412500,2023-10-11,30.0,70,2327877.0,0,0.0,21.0,0.0,0.0,6.983631e+05,1.561146e+07,1.068307e+07
39706,17486356,2025-12-18,4.0,53,1288459.0,0,0.0,26.0,0.0,0.0,6.320742e+05,1.412962e+07,9.669031e+06
39707,13992059,2024-03-04,25.0,259,6629961.0,0,0.0,197.0,0.0,0.0,5.042866e+06,1.127301e+08,7.714225e+07
39708,17496297,2025-12-18,4.0,57,9167048.0,0,0.0,20.0,0.0,0.0,3.216508e+06,7.190298e+07,4.920390e+07


In [145]:
fact_data_year_not_thu_ho[fact_data_year_not_thu_ho['gmv_med'] >= 30000000]

,cus_id,ngay_hoptac,thoigian_sd,tong_don,tong_tien,tongdon_cod,thuho_tongdon,don_ptc,don_ptc_cod,thu_ho,chi_phi_ptc,gmv_med,gmv_mean
0,13845857,2024-01-26,27.0,573,7193095.0,0,0.0,349.0,0.0,0.0,4.381135e+06,9.793747e+07,6.701954e+07
1,12627701,2023-03-08,37.0,1388,54142752.0,0,0.0,1188.0,0.0,0.0,4.634120e+07,1.035928e+09,7.088954e+08
2,7551727,2020-03-13,73.0,440,21306919.0,0,0.0,239.0,0.0,0.0,1.157353e+07,2.587189e+08,1.770438e+08
3,9216277,2021-05-07,59.0,283,4337732.0,0,0.0,148.0,0.0,0.0,2.268496e+06,5.071078e+07,3.470187e+07
4,12105262,2022-09-27,43.0,911,10579194.0,0,0.0,596.0,0.0,0.0,6.921185e+06,1.547187e+08,1.058755e+08
...,...,...,...,...,...,...,...,...,...,...,...,...,...
39691,17441116,2025-12-11,4.0,356,21494104.0,0,0.0,183.0,0.0,0.0,1.104894e+07,2.469919e+08,1.690189e+08
39693,14237707,2024-05-03,23.0,46,7895716.0,0,0.0,19.0,0.0,0.0,3.261274e+06,7.290370e+07,4.988870e+07
39698,14931905,2024-08-26,20.0,349,4814597.0,0,0.0,168.0,0.0,0.0,2.317628e+06,5.180910e+07,3.545346e+07
39707,13992059,2024-03-04,25.0,259,6629961.0,0,0.0,197.0,0.0,0.0,5.042866e+06,1.127301e+08,7.714225e+07


In [151]:
not_cod_med = fact_data_year_not_thu_ho
not_cod_med = fact_data_year_not_thu_ho.drop(columns = ['gmv_mean'])
not_cod_med.rename(columns = {'gmv_med': 'gmv'}, inplace = True)

In [152]:
not_cod_mean = fact_data_year_not_thu_ho
not_cod_mean = fact_data_year_not_thu_ho.drop(columns = ['gmv_med'])
not_cod_mean.rename(columns = {'gmv_mean': 'gmv'}, inplace = True)

In [157]:
not_cod_med

,cus_id,ngay_hoptac,thoigian_sd,tong_don,tong_tien,tongdon_cod,thuho_tongdon,don_ptc,don_ptc_cod,thu_ho,chi_phi_ptc,gmv
0,13845857,2024-01-26,27.0,573,7193095.0,0,0.0,349.0,0.0,0.0,4.381135e+06,9.793747e+07
1,12627701,2023-03-08,37.0,1388,54142752.0,0,0.0,1188.0,0.0,0.0,4.634120e+07,1.035928e+09
2,7551727,2020-03-13,73.0,440,21306919.0,0,0.0,239.0,0.0,0.0,1.157353e+07,2.587189e+08
3,9216277,2021-05-07,59.0,283,4337732.0,0,0.0,148.0,0.0,0.0,2.268496e+06,5.071078e+07
4,12105262,2022-09-27,43.0,911,10579194.0,0,0.0,596.0,0.0,0.0,6.921185e+06,1.547187e+08
...,...,...,...,...,...,...,...,...,...,...,...,...
39705,13412500,2023-10-11,30.0,70,2327877.0,0,0.0,21.0,0.0,0.0,6.983631e+05,1.561146e+07
39706,17486356,2025-12-18,4.0,53,1288459.0,0,0.0,26.0,0.0,0.0,6.320742e+05,1.412962e+07
39707,13992059,2024-03-04,25.0,259,6629961.0,0,0.0,197.0,0.0,0.0,5.042866e+06,1.127301e+08
39708,17496297,2025-12-18,4.0,57,9167048.0,0,0.0,20.0,0.0,0.0,3.216508e+06,7.190298e+07


#### Tổng hợp

In [147]:
# có cod:


In [154]:
cod = fact_data_year_thu_ho.drop(columns = ['rev_per_don_ptc','rev_per_tongdon','%chi_phi_ptc_case2'])
cod.rename(columns = {'gmv_case2': 'gmv'}, inplace = True)
cod

,cus_id,ngay_hoptac,thoigian_sd,tong_don,tong_tien,tongdon_cod,thuho_tongdon,don_ptc,don_ptc_cod,thu_ho,chi_phi_ptc,gmv
0,3815832,2019-01-01,87.0,323,11286708.0,50,1.672405e+08,160.0,33.0,9.652563e+07,5.590939e+06,4.680030e+08
1,13552722,2023-11-14,29.0,197,23608628.0,4,7.400000e+06,82.0,1.0,3.420000e+06,9.826942e+06,2.804400e+08
2,7976779,2020-06-18,70.0,764,42656147.0,86,2.461992e+07,600.0,73.0,1.085998e+07,3.349959e+07,8.926013e+07
3,14988133,2024-09-03,19.0,203,6339667.0,98,3.282350e+07,65.0,28.0,1.528300e+07,2.029943e+06,3.547839e+07
4,14067926,2024-03-22,25.0,3123,88299650.0,1731,2.532835e+09,2728.0,1497.0,2.113407e+09,7.713143e+07,3.851286e+09
...,...,...,...,...,...,...,...,...,...,...,...,...
87496,10185158,2021-09-24,55.0,80,1871217.0,65,1.856000e+07,44.0,35.0,9.405000e+06,1.029169e+06,1.182343e+07
87497,7104910,2019-11-08,77.0,43,2726078.0,13,5.951700e+07,15.0,3.0,1.603500e+07,9.509574e+05,8.017500e+07
87498,17495758,2025-12-18,4.0,270,5221779.0,219,8.165001e+07,167.0,135.0,5.082301e+07,3.229767e+06,6.286994e+07
87499,10669683,2021-11-23,53.0,35,1892695.0,20,2.452510e+07,10.0,5.0,6.801000e+06,5.407700e+05,1.360200e+07


In [ ]:
### all theo med

In [153]:
data = pd.concat([cod,not_cod_med])
data

,cus_id,ngay_hoptac,thoigian_sd,tong_don,tong_tien,tongdon_cod,thuho_tongdon,don_ptc,don_ptc_cod,thu_ho,chi_phi_ptc,gmv
0,3815832,2019-01-01,87.0,323,11286708.0,50,1.672405e+08,160.0,33.0,9.652563e+07,5.590939e+06,4.680030e+08
1,13552722,2023-11-14,29.0,197,23608628.0,4,7.400000e+06,82.0,1.0,3.420000e+06,9.826942e+06,2.804400e+08
2,7976779,2020-06-18,70.0,764,42656147.0,86,2.461992e+07,600.0,73.0,1.085998e+07,3.349959e+07,8.926013e+07
3,14988133,2024-09-03,19.0,203,6339667.0,98,3.282350e+07,65.0,28.0,1.528300e+07,2.029943e+06,3.547839e+07
4,14067926,2024-03-22,25.0,3123,88299650.0,1731,2.532835e+09,2728.0,1497.0,2.113407e+09,7.713143e+07,3.851286e+09
...,...,...,...,...,...,...,...,...,...,...,...,...
39705,13412500,2023-10-11,30.0,70,2327877.0,0,0.000000e+00,21.0,0.0,0.000000e+00,6.983631e+05,1.561146e+07
39706,17486356,2025-12-18,4.0,53,1288459.0,0,0.000000e+00,26.0,0.0,0.000000e+00,6.320742e+05,1.412962e+07
39707,13992059,2024-03-04,25.0,259,6629961.0,0,0.000000e+00,197.0,0.0,0.000000e+00,5.042866e+06,1.127301e+08
39708,17496297,2025-12-18,4.0,57,9167048.0,0,0.000000e+00,20.0,0.0,0.000000e+00,3.216508e+06,7.190298e+07


In [158]:
import numpy as np
import pandas as pd

bins = (
    [-np.inf, 10000000] +
    list(range(10000000, 105000000, 5000000)) +
    list(range(100000000, 1050000000, 50000000)) +
    list(range(1000000000, 11000000000, 1000000000)) +
    [np.inf]
)
labels = []

labels.append("<10")

# 10–100 step 5
labels += [f"{i}-{i+5}" for i in range(10, 100, 5)]

# 100–1000 step 50
labels += [f"{i}-{i+50}" for i in range(100, 1000, 50)]

# 1000–10000 step 1000
labels += [f"{i}-{i+1000}" for i in range(1000, 10000, 1000)]

labels.append(">10000")


data['gmv_bucket'] = pd.cut(
    data['gmv'],
    bins=bins,
    labels=labels,
    duplicates='drop'
)

In [159]:
data

,cus_id,ngay_hoptac,thoigian_sd,tong_don,tong_tien,tongdon_cod,thuho_tongdon,don_ptc,don_ptc_cod,thu_ho,chi_phi_ptc,gmv,gmv_bucket
0,3815832,2019-01-01,87.0,323,11286708.0,50,1.672405e+08,160.0,33.0,9.652563e+07,5.590939e+06,4.680030e+08,450-500
1,13552722,2023-11-14,29.0,197,23608628.0,4,7.400000e+06,82.0,1.0,3.420000e+06,9.826942e+06,2.804400e+08,250-300
2,7976779,2020-06-18,70.0,764,42656147.0,86,2.461992e+07,600.0,73.0,1.085998e+07,3.349959e+07,8.926013e+07,85-90
3,14988133,2024-09-03,19.0,203,6339667.0,98,3.282350e+07,65.0,28.0,1.528300e+07,2.029943e+06,3.547839e+07,35-40
4,14067926,2024-03-22,25.0,3123,88299650.0,1731,2.532835e+09,2728.0,1497.0,2.113407e+09,7.713143e+07,3.851286e+09,3000-4000
...,...,...,...,...,...,...,...,...,...,...,...,...,...
39705,13412500,2023-10-11,30.0,70,2327877.0,0,0.000000e+00,21.0,0.0,0.000000e+00,6.983631e+05,1.561146e+07,15-20
39706,17486356,2025-12-18,4.0,53,1288459.0,0,0.000000e+00,26.0,0.0,0.000000e+00,6.320742e+05,1.412962e+07,10-15
39707,13992059,2024-03-04,25.0,259,6629961.0,0,0.000000e+00,197.0,0.0,0.000000e+00,5.042866e+06,1.127301e+08,100-150
39708,17496297,2025-12-18,4.0,57,9167048.0,0,0.000000e+00,20.0,0.0,0.000000e+00,3.216508e+06,7.190298e+07,70-75


In [160]:
data.to_csv(r'../fact_data_year_thu_ho.csv', index = False)

In [146]:
31762 + 77100 

108862

In [141]:
1.600441e+08

160044100.0

In [143]:
fact_data_year_thu_ho[fact_data_year_thu_ho['gmv_case2'] >= 30000000]

,cus_id,ngay_hoptac,thoigian_sd,tong_don,tong_tien,tongdon_cod,thuho_tongdon,don_ptc,don_ptc_cod,thu_ho,chi_phi_ptc,rev_per_don_ptc,rev_per_tongdon,gmv_case2,%chi_phi_ptc_case2
0,3815832,2019-01-01,87.0,323,11286708.0,50,1.672405e+08,160.0,33.0,9.652563e+07,5.590939e+06,2.925019e+06,3.344809e+06,4.680030e+08,0.011946
1,13552722,2023-11-14,29.0,197,23608628.0,4,7.400000e+06,82.0,1.0,3.420000e+06,9.826942e+06,3.420000e+06,1.850000e+06,2.804400e+08,0.035041
2,7976779,2020-06-18,70.0,764,42656147.0,86,2.461992e+07,600.0,73.0,1.085998e+07,3.349959e+07,1.487669e+05,2.862781e+05,8.926013e+07,0.375303
3,14988133,2024-09-03,19.0,203,6339667.0,98,3.282350e+07,65.0,28.0,1.528300e+07,2.029943e+06,5.458214e+05,3.349337e+05,3.547839e+07,0.057216
4,14067926,2024-03-22,25.0,3123,88299650.0,1731,2.532835e+09,2728.0,1497.0,2.113407e+09,7.713143e+07,1.411762e+06,1.463221e+06,3.851286e+09,0.020027
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
87493,13590123,2023-11-24,29.0,235,5684977.0,224,6.896500e+07,129.0,127.0,3.193500e+07,3.120690e+06,2.514567e+05,3.078795e+05,3.243791e+07,0.096205
87494,7785708,2020-04-28,72.0,225,8592301.0,119,6.547684e+07,135.0,69.0,3.644634e+07,5.155381e+06,5.282078e+05,5.502256e+05,7.130806e+07,0.072297
87497,7104910,2019-11-08,77.0,43,2726078.0,13,5.951700e+07,15.0,3.0,1.603500e+07,9.509574e+05,5.345000e+06,4.578231e+06,8.017500e+07,0.011861
87498,17495758,2025-12-18,4.0,270,5221779.0,219,8.165001e+07,167.0,135.0,5.082301e+07,3.229767e+06,3.764667e+05,3.728311e+05,6.286994e+07,0.051372


In [133]:
fact_data_year_thu_ho[fact_data_year_thu_ho['gmv_case2'] >= 30000000]['%chi_phi_ptc_case2'].describe(
    percentiles=[i/100 for i in range(5, 100, 5)]
)

count    77100.000000
mean         0.065371
std          0.132338
min          0.000562
5%           0.012902
10%          0.017247
15%          0.020972
20%          0.024257
25%          0.027508
30%          0.030793
35%          0.034065
40%          0.037470
45%          0.041007
50%          0.044734
55%          0.048912
60%          0.053509
65%          0.058795
70%          0.065042
75%          0.072472
80%          0.082063
85%          0.095309
90%          0.115072
95%          0.155992
max         10.534368
Name: %chi_phi_ptc_case2, dtype: float64

In [134]:
0.065371

0.065371

In [122]:
719690.4761904762 * 39

28067928.57142857

In [48]:
fact_data_year_thu_ho.to_csv(r'../fact_data_year_thu_ho.csv', index = False)

In [123]:
fact_data_year_thu_ho['%chi_phi_ptc_case2'].describe(
    percentiles=[i/100 for i in range(5, 100, 5)]
)

count    8.750100e+04
mean     2.096734e+01
std      4.651803e+03
min      5.615531e-04
5%       1.355887e-02
10%      1.825661e-02
15%      2.217570e-02
20%      2.580010e-02
25%      2.937108e-02
30%      3.291724e-02
35%      3.654659e-02
40%      4.027325e-02
45%      4.420958e-02
50%      4.849015e-02
55%      5.313962e-02
60%      5.848514e-02
65%      6.459641e-02
70%      7.183617e-02
75%      8.093535e-02
80%      9.288379e-02
85%      1.092193e-01
90%      1.368511e-01
95%      2.105479e-01
max      1.365288e+06
Name: %chi_phi_ptc_case2, dtype: float64

In [127]:
fact_data_year_thu_ho[fact_data_year_thu_ho['%chi_phi_ptc_case2'] >= 1 ]

,cus_id,ngay_hoptac,thoigian_sd,tong_don,tong_tien,tongdon_cod,thuho_tongdon,don_ptc,don_ptc_cod,thu_ho,chi_phi_ptc,rev_per_don_ptc,rev_per_tongdon,gmv_case2,%chi_phi_ptc_case2
27,11073475,2022-02-07,50.0,1254,1.109575e+08,378,29130000.0,1172.0,370.0,28185000.0,1.037019e+08,7.617568e+04,7.706349e+04,8.927789e+07,1.161563
129,11343240,2022-03-28,49.0,4212,1.531172e+08,1,31000.0,3751.0,1.0,31000.0,1.363586e+08,3.100000e+04,3.100000e+04,1.162810e+08,1.172665
175,367692,2017-10-02,102.0,360,1.631802e+08,22,14823000.0,218.0,19.0,8018000.0,9.881468e+07,4.220000e+05,6.737727e+05,9.199600e+07,1.074119
234,186363,2017-09-28,103.0,525,1.820384e+09,3,5172000.0,356.0,1.0,1415000.0,1.234394e+09,1.415000e+06,1.724000e+06,5.037400e+08,2.450459
327,7127406,2019-11-20,77.0,1604,1.808620e+07,61,61000.0,1604.0,61.0,61000.0,1.808620e+07,1.000000e+03,1.000000e+03,1.604000e+06,11.275686
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
87338,5336734,2019-01-16,87.0,65,7.915006e+07,8,160000.0,35.0,5.0,100000.0,4.261927e+07,2.000000e+04,2.000000e+04,7.000000e+05,60.884665
87342,17255650,2025-11-12,5.0,7794,7.879750e+07,7793,9790000.0,7231.0,7230.0,7429000.0,7.310556e+07,1.027524e+03,1.256256e+03,7.430028e+06,9.839204
87459,11909910,2022-07-28,45.0,316,1.189486e+08,161,55206000.0,201.0,109.0,27665000.0,7.566037e+07,2.538073e+05,3.428944e+05,5.101528e+07,1.483093
87468,17409152,2025-12-04,4.0,222,9.935322e+06,9,275000.0,89.0,3.0,80000.0,3.983080e+06,2.666667e+04,3.055556e+04,2.373333e+06,1.678264


In [132]:
7.000000e+05	

700000.0

In [131]:
100000.0 / 5

20000.0

In [54]:
fact_data_year_thu_ho['%chi_phi_ptc_case2'].describe(
    percentiles=[i/100 for i in range(5, 100, 5)]
)

count    3.895870e+05
mean     1.619831e+01
std      5.395857e+03
min     -2.007976e-05
5%       1.356554e-02
10%      1.915007e-02
15%      2.410654e-02
20%      2.892390e-02
25%      3.374386e-02
30%      3.869477e-02
35%      4.381898e-02
40%      4.936645e-02
45%      5.532470e-02
50%      6.213039e-02
55%      6.970356e-02
60%      7.846521e-02
65%      8.875122e-02
70%      1.013041e-01
75%      1.171307e-01
80%      1.387277e-01
85%      1.712211e-01
90%      2.303872e-01
95%      4.064819e-01
max      2.977133e+06
Name: %chi_phi_ptc_case2, dtype: float64

In [62]:
fact_data_year_thu_ho['gmv_case2'].describe(
    percentiles=[i/100 for i in range(5, 100, 5)]
)

count    3.895870e+05
mean     3.673383e+08
std      1.090221e+10
min      1.000000e+00
5%       2.130000e+05
10%      4.089556e+05
15%      6.750000e+05
20%      1.020000e+06
25%      1.520000e+06
30%      2.244000e+06
35%      3.270000e+06
40%      4.822701e+06
45%      7.170000e+06
50%      1.077508e+07
55%      1.634473e+07
60%      2.514610e+07
65%      3.904733e+07
70%      6.227894e+07
75%      1.012104e+08
80%      1.698955e+08
85%      3.003996e+08
90%      5.747626e+08
95%      1.345872e+09
max      5.950514e+12
Name: gmv_case2, dtype: float64

In [67]:
import numpy as np
import pandas as pd

bins = (
    [-np.inf, 10000000] +
    list(range(10000000, 105000000, 5000000)) +
    list(range(100000000, 1050000000, 50000000)) +
    list(range(1000000000, 11000000000, 1000000000)) +
    [np.inf]
)
labels = []

labels.append("<10")

# 10–100 step 5
labels += [f"{i}-{i+5}" for i in range(10, 100, 5)]

# 100–1000 step 50
labels += [f"{i}-{i+50}" for i in range(100, 1000, 50)]

# 1000–10000 step 1000
labels += [f"{i}-{i+1000}" for i in range(1000, 10000, 1000)]

labels.append(">10000")


fact_data_year_thu_ho['gmv_bucket'] = pd.cut(
    fact_data_year_thu_ho['gmv_case2'],
    bins=bins,
    labels=labels,
    duplicates='drop'
)

In [68]:
fact_data_year_thu_ho

,cus_id,ngay_hoptac,thoigian_sd,tong_don,tong_tien,tongdon_cod,thuho_tongdon,don_ptc,don_ptc_cod,thu_ho,rev_per_don_ptc,rev_per_tongdon,gmv_case1,gmv_case2,chi_phi_ptc,%chi_phi_ptc_case1,%chi_phi_ptc_case2,gmv_bucket
0,3983710,2019-01-03,87.0,264,10916999.0,194,70801250.0,132.0,100.0,36688000.0,3.668800e+05,3.649549e+05,4.830110e+07,4.842816e+07,5.458500e+06,0.113010,0.112713,45-50
1,15409953,2024-11-27,17.0,884,50582582.0,802,880883000.0,786.0,725.0,788215000.0,1.087193e+06,1.098358e+06,8.589215e+08,8.545338e+08,4.497501e+07,0.052362,0.052631,850-900
2,3815832,2019-01-01,87.0,323,11286708.0,50,167240460.0,160.0,33.0,96525628.0,2.925019e+06,3.344809e+06,5.015863e+08,4.680030e+08,5.590939e+06,0.011147,0.011946,450-500
3,15801244,2025-02-25,14.0,92,3191905.0,78,100712860.0,27.0,25.0,32252260.0,1.290090e+06,1.291191e+06,3.484729e+07,3.483244e+07,9.367547e+05,0.026882,0.026893,30-35
4,13552722,2023-11-14,29.0,197,23608628.0,4,7400000.0,82.0,1.0,3420000.0,3.420000e+06,1.850000e+06,2.160700e+08,2.804400e+08,9.826942e+06,0.045480,0.035041,250-300
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
389582,18157102,2026-03-31,1.0,2,56501.0,1,350000.0,1.0,0.0,0.0,0.000000e+00,3.500000e+05,1.750000e+05,3.500000e+05,2.825050e+04,0.161431,0.080716,<10
389583,18040605,2026-03-13,1.0,12,198000.0,9,1446000.0,5.0,5.0,538000.0,1.076000e+05,1.606667e+05,6.706667e+05,5.380000e+05,8.250000e+04,0.123012,0.153346,<10
389584,18143724,2026-03-31,1.0,23,538833.0,22,36500000.0,1.0,0.0,0.0,0.000000e+00,1.659091e+06,8.295455e+05,1.659091e+06,2.342752e+04,0.028241,0.014121,<10
389585,18029623,2026-03-15,1.0,1,19651.0,1,150000.0,1.0,1.0,150000.0,1.500000e+05,1.500000e+05,1.500000e+05,1.500000e+05,1.965100e+04,0.131007,0.131007,<10


In [69]:
fact_data_year_thu_ho.to_csv(r'../fact_data_year_thu_ho.csv', index = False)

In [56]:
fact_data_year_thu_ho[fact_data_year_thu_ho['%chi_phi_ptc_case1'] < 0] 

,cus_id,ngay_hoptac,thoigian_sd,tong_don,tong_tien,tongdon_cod,thuho_tongdon,don_ptc,don_ptc_cod,thu_ho,rev_per_don_ptc,rev_per_tongdon,gmv_case1,gmv_case2,chi_phi_ptc,%chi_phi_ptc_case1,%chi_phi_ptc_case2
121737,15518793,2024-12-18,16.0,941,-18514.0,0,0.0,26.0,24.0,23516000.0,979833.333333,0.0,1.273783e+07,2.547567e+07,-511.545165,-0.00004,-0.00002


In [72]:
fact_data_year_thu_ho[(fact_data_year_thu_ho['thoigian_sd'] >= 4) & (fact_data_year_thu_ho['gmv_case2'] >= 30000000)]['%chi_phi_ptc_case2'].describe(
    percentiles=[i/100 for i in range(5, 100, 5)]
)

count    143124.000000
mean          0.063168
std           0.120014
min           0.000123
5%            0.010864
10%           0.015331
15%           0.019018
20%           0.022453
25%           0.025815
30%           0.029234
35%           0.032698
40%           0.036329
45%           0.040087
50%           0.044077
55%           0.048413
60%           0.053241
65%           0.058816
70%           0.065306
75%           0.073077
80%           0.083028
85%           0.096462
90%           0.116395
95%           0.155856
max          12.551031
Name: %chi_phi_ptc_case2, dtype: float64

##### Nhóm 2. Không có thu hộ

In [73]:
fact_data_year

,cus_id,ngay_hoptac,thoigian_sd,tong_don,tong_tien,tongdon_cod,thuho_tongdon,don_ptc,don_ptc_cod,thu_ho
0,15521383,2024-12-19,16.0,23,592803.0,0,0.0,2.0,0.0,0.0
1,3983710,2019-01-03,87.0,264,10916999.0,194,70801250.0,132.0,100.0,36688000.0
2,15888055,2025-03-13,13.0,516,11373390.0,0,0.0,227.0,0.0,0.0
3,15409953,2024-11-27,17.0,884,50582582.0,802,880883000.0,786.0,725.0,788215000.0
5,3815832,2019-01-01,87.0,323,11286708.0,50,167240460.0,160.0,33.0,96525628.0
...,...,...,...,...,...,...,...,...,...,...
2521898,13867217,2024-01-30,27.0,2,52101.0,0,0.0,1.0,0.0,0.0
2522191,18096661,2026-03-21,1.0,1,18150.0,0,0.0,1.0,0.0,0.0
2522294,18134956,2026-03-30,1.0,1,8030000.0,0,0.0,1.0,0.0,0.0
2522366,18144752,2026-03-31,1.0,1,18750.0,0,0.0,1.0,0.0,0.0


In [74]:
fact_data_year_not_thu_ho = fact_data_year[(fact_data_year['thu_ho'] == 0) & (fact_data_year['thuho_tongdon'] == 0)].reset_index(drop = True)
fact_data_year_not_thu_ho

,cus_id,ngay_hoptac,thoigian_sd,tong_don,tong_tien,tongdon_cod,thuho_tongdon,don_ptc,don_ptc_cod,thu_ho
0,15521383,2024-12-19,16.0,23,592803.0,0,0.0,2.0,0.0,0.0
1,15888055,2025-03-13,13.0,516,11373390.0,0,0.0,227.0,0.0,0.0
2,13845857,2024-01-26,27.0,573,7193095.0,0,0.0,349.0,0.0,0.0
3,13391815,2023-10-03,30.0,5,257609.0,0,0.0,1.0,0.0,0.0
4,1850115,2018-09-04,91.0,33,1268905.0,0,0.0,2.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...
312601,13867217,2024-01-30,27.0,2,52101.0,0,0.0,1.0,0.0,0.0
312602,18096661,2026-03-21,1.0,1,18150.0,0,0.0,1.0,0.0,0.0
312603,18134956,2026-03-30,1.0,1,8030000.0,0,0.0,1.0,0.0,0.0
312604,18144752,2026-03-31,1.0,1,18750.0,0,0.0,1.0,0.0,0.0


In [76]:
fact_data_year_not_thu_ho['chi_phi_ptc'] = fact_data_year_not_thu_ho['don_ptc'] * ( fact_data_year_not_thu_ho['tong_tien'] / fact_data_year_not_thu_ho['tong_don'] ) 

##### Xác định tỉ lệ % chi phí vtp cho lượng đơn ptc là trung vị của tệp tgsd >= 4, gmv >= 30tr : 4.4%

In [80]:
fact_data_year_not_thu_ho['gmv_meadian'] = fact_data_year_not_thu_ho['chi_phi_ptc'] / 0.04

##### Xác định tỉ lệ % chi phí vtp cho lượng đơn ptc là trung bình của tệp tgsd >= 4, gmv >= 30tr : 6.3%

In [81]:
fact_data_year_not_thu_ho['gmv_mean'] = fact_data_year_not_thu_ho['chi_phi_ptc'] / 0.063

In [82]:
fact_data_year_not_thu_ho

,cus_id,ngay_hoptac,thoigian_sd,tong_don,tong_tien,tongdon_cod,thuho_tongdon,don_ptc,don_ptc_cod,thu_ho,chi_phi_ptc,gmv_meadian,gmv_mean
0,15521383,2024-12-19,16.0,23,592803.0,0,0.0,2.0,0.0,0.0,5.154809e+04,1.288702e+06,8.182236e+05
1,15888055,2025-03-13,13.0,516,11373390.0,0,0.0,227.0,0.0,0.0,5.003410e+06,1.250852e+08,7.941921e+07
2,13845857,2024-01-26,27.0,573,7193095.0,0,0.0,349.0,0.0,0.0,4.381135e+06,1.095284e+08,6.954182e+07
3,13391815,2023-10-03,30.0,5,257609.0,0,0.0,1.0,0.0,0.0,5.152180e+04,1.288045e+06,8.178063e+05
4,1850115,2018-09-04,91.0,33,1268905.0,0,0.0,2.0,0.0,0.0,7.690333e+04,1.922583e+06,1.220688e+06
...,...,...,...,...,...,...,...,...,...,...,...,...,...
312601,13867217,2024-01-30,27.0,2,52101.0,0,0.0,1.0,0.0,0.0,2.605050e+04,6.512625e+05,4.135000e+05
312602,18096661,2026-03-21,1.0,1,18150.0,0,0.0,1.0,0.0,0.0,1.815000e+04,4.537500e+05,2.880952e+05
312603,18134956,2026-03-30,1.0,1,8030000.0,0,0.0,1.0,0.0,0.0,8.030000e+06,2.007500e+08,1.274603e+08
312604,18144752,2026-03-31,1.0,1,18750.0,0,0.0,1.0,0.0,0.0,1.875000e+04,4.687500e+05,2.976190e+05


In [85]:
fact_data_year_not_thu_ho[(fact_data_year_not_thu_ho['thoigian_sd'] >= 4) & (fact_data_year_not_thu_ho['gmv_meadian'] >= 30000000)]['gmv_meadian'].describe(
    percentiles=[i/100 for i in range(5, 100, 5)]
)

count    5.678500e+04
mean     6.473540e+08
std      2.118443e+10
min      3.000000e+07
5%       3.315072e+07
10%      3.668385e+07
15%      4.098319e+07
20%      4.588642e+07
25%      5.164790e+07
30%      5.837339e+07
35%      6.658791e+07
40%      7.609379e+07
45%      8.774644e+07
50%      1.026156e+08
55%      1.208936e+08
60%      1.441496e+08
65%      1.759952e+08
70%      2.160640e+08
75%      2.725580e+08
80%      3.598473e+08
85%      5.043633e+08
90%      7.794474e+08
95%      1.523459e+09
max      4.683872e+12
Name: gmv_meadian, dtype: float64

In [87]:
fact_data_year_not_thu_ho[(fact_data_year_not_thu_ho['thoigian_sd'] >= 4) & (fact_data_year_not_thu_ho['gmv_meadian'] >= 30000000)]

,cus_id,ngay_hoptac,thoigian_sd,tong_don,tong_tien,tongdon_cod,thuho_tongdon,don_ptc,don_ptc_cod,thu_ho,chi_phi_ptc,gmv_meadian,gmv_mean
1,15888055,2025-03-13,13.0,516,11373390.0,0,0.0,227.0,0.0,0.0,5.003410e+06,1.250852e+08,7.941921e+07
2,13845857,2024-01-26,27.0,573,7193095.0,0,0.0,349.0,0.0,0.0,4.381135e+06,1.095284e+08,6.954182e+07
8,12627701,2023-03-08,37.0,1388,54142752.0,0,0.0,1188.0,0.0,0.0,4.634120e+07,1.158530e+09,7.355746e+08
10,7551727,2020-03-13,73.0,440,21306919.0,0,0.0,239.0,0.0,0.0,1.157353e+07,2.893383e+08,1.837068e+08
14,5995403,2019-01-19,87.0,78,20110182.0,0,0.0,21.0,0.0,0.0,5.414280e+06,1.353570e+08,8.594095e+07
...,...,...,...,...,...,...,...,...,...,...,...,...,...
312450,14309142,2024-05-24,23.0,1,2335000.0,0,0.0,1.0,0.0,0.0,2.335000e+06,5.837500e+07,3.706349e+07
312517,2051543,2018-11-01,89.0,11,54016500.0,0,0.0,11.0,0.0,0.0,5.401650e+07,1.350412e+09,8.574048e+08
312522,12943244,2023-06-09,34.0,25,54000000.0,0,0.0,25.0,0.0,0.0,5.400000e+07,1.350000e+09,8.571429e+08
312551,15366388,2024-11-18,17.0,1,5625000.0,0,0.0,1.0,0.0,0.0,5.625000e+06,1.406250e+08,8.928571e+07


In [88]:
fact_data_year_not_thu_ho[(fact_data_year_not_thu_ho['thoigian_sd'] >= 4) & (fact_data_year_not_thu_ho['gmv_mean'] >= 30000000)]

,cus_id,ngay_hoptac,thoigian_sd,tong_don,tong_tien,tongdon_cod,thuho_tongdon,don_ptc,don_ptc_cod,thu_ho,chi_phi_ptc,gmv_meadian,gmv_mean
1,15888055,2025-03-13,13.0,516,11373390.0,0,0.0,227.0,0.0,0.0,5.003410e+06,1.250852e+08,7.941921e+07
2,13845857,2024-01-26,27.0,573,7193095.0,0,0.0,349.0,0.0,0.0,4.381135e+06,1.095284e+08,6.954182e+07
8,12627701,2023-03-08,37.0,1388,54142752.0,0,0.0,1188.0,0.0,0.0,4.634120e+07,1.158530e+09,7.355746e+08
10,7551727,2020-03-13,73.0,440,21306919.0,0,0.0,239.0,0.0,0.0,1.157353e+07,2.893383e+08,1.837068e+08
14,5995403,2019-01-19,87.0,78,20110182.0,0,0.0,21.0,0.0,0.0,5.414280e+06,1.353570e+08,8.594095e+07
...,...,...,...,...,...,...,...,...,...,...,...,...,...
312450,14309142,2024-05-24,23.0,1,2335000.0,0,0.0,1.0,0.0,0.0,2.335000e+06,5.837500e+07,3.706349e+07
312517,2051543,2018-11-01,89.0,11,54016500.0,0,0.0,11.0,0.0,0.0,5.401650e+07,1.350412e+09,8.574048e+08
312522,12943244,2023-06-09,34.0,25,54000000.0,0,0.0,25.0,0.0,0.0,5.400000e+07,1.350000e+09,8.571429e+08
312551,15366388,2024-11-18,17.0,1,5625000.0,0,0.0,1.0,0.0,0.0,5.625000e+06,1.406250e+08,8.928571e+07


In [89]:
1.250852e+08

125085200.0

In [90]:
7.941921e+07

79419210.0

In [83]:
fact_data_year_not_thu_ho['gmv_meadian'].describe(
    percentiles=[i/100 for i in range(5, 100, 5)]
)

count    3.126060e+05
mean     1.227697e+08
std      9.032437e+09
min      0.000000e+00
5%       5.669000e+05
10%      6.905050e+05
15%      7.914354e+05
20%      9.146192e+05
25%      1.099984e+06
30%      1.347215e+06
35%      1.617294e+06
40%      1.988191e+06
45%      2.471632e+06
50%      3.137908e+06
55%      4.070564e+06
60%      5.419923e+06
65%      7.483397e+06
70%      1.074551e+07
75%      1.615655e+07
80%      2.579724e+07
85%      4.481280e+07
90%      9.029556e+07
95%      2.469917e+08
max      4.683872e+12
Name: gmv_meadian, dtype: float64

In [14]:
fact_data_year['thoigian_sd'].value_counts().sort_index()

thoigian_sd
1.0      63054
2.0      32576
3.0      76381
4.0      82324
5.0      78396
         ...  
99.0      1538
100.0     1452
101.0     1450
102.0     1602
103.0    22652
Name: count, Length: 103, dtype: int64

In [15]:
fact_data_vip = fact_data_year[(fact_data_year['tong_don'] >= 900) |(fact_data_year['tong_tien'] >= 100000000) ]
fact_data_vip

,cus_id,tgsd_flag,tong_don,tong_tien,tongdon_cod,thuho_tongdon,don_ptc,don_ptc_cod,thu_ho
13,125,1,904,201328805.0,4,1.715000e+06,763.0,2.0,880000.0
39,427,1,1311,21037446.0,142,1.074590e+08,1113.0,127.0,94837000.0
50,538,1,1816,32280758.0,0,0.000000e+00,1564.0,0.0,0.0
64,734,1,13363,602716337.0,1,1.000000e+06,12708.0,1.0,1000000.0
65,781,1,1497,37902189.0,1011,5.190585e+08,478.0,330.0,168348600.0
...,...,...,...,...,...,...,...,...,...
2515087,18123209,0,5179,111038720.0,5128,2.140454e+09,1494.0,1485.0,602033001.0
2515532,18124420,0,10000,145000000.0,0,0.000000e+00,109.0,0.0,0.0
2516135,18126571,0,10534,79030416.0,0,0.000000e+00,0.0,0.0,0.0
2516947,18129176,0,2548,36748250.0,0,0.000000e+00,183.0,0.0,0.0


#### 2. Khách hàng hợp tác với vtp ngày đầu tiên >= 6 tháng

In [16]:
temp = fact_data_year[~fact_data_year['cus_id'].isin(fact_data_vip['cus_id'].unique())]
rule_dataset2 = temp[temp['tgsd_flag'] == 1]
rule_dataset2.shape

(2137313, 9)

#### 3. Trong 6 tháng gần nhất, khách hàng không có tháng nào không có đơn ptc

In [17]:
# Lấy đúng 6 tháng gần nhất
last_6_months = sorted(fact_data['month'].unique())[-6:]

# Lọc data
check_data = fact_data[fact_data['month'].isin(last_6_months)].copy()

# Pivot
pivot_df = check_data.pivot_table(
    index='cus_id', 
    columns='month', 
    values='don_ptc', 
    aggfunc='sum'
)

# Đảm bảo đủ 6 tháng
pivot_df = pivot_df.reindex(columns=last_6_months).fillna(0)

# Khách có ít nhất 1 tháng không có đơn
cus_co_thang_trong = pivot_df[(pivot_df == 0).any(axis=1)]

# List ID
list_cus_trong = cus_co_thang_trong.index.tolist()

cus_pass = pivot_df[(pivot_df != 0).all(axis=1)]
list_cus_pass = cus_pass.index.tolist()

total_cus = pivot_df.shape[0]
num_cus_trong = len(list_cus_trong)
num_cus_full = len(list_cus_pass)

print("Tổng KH:", total_cus)
print("KH có tháng trống:", num_cus_trong)
print("KH đủ 6 tháng:", num_cus_full)

Tổng KH: 1828000
KH có tháng trống: 1719518
KH đủ 6 tháng: 108482


In [18]:
fact_data_year_pass = rule_dataset2[rule_dataset2['cus_id'].isin(list_cus_pass)]
fact_data_year_pass

,cus_id,tgsd_flag,tong_don,tong_tien,tongdon_cod,thuho_tongdon,don_ptc,don_ptc_cod,thu_ho
2,20,1,764,16775286.0,297,219789000.0,627.0,248.0,174743000.0
9,78,1,121,4872605.0,22,13459000.0,34.0,8.0,5417000.0
11,112,1,313,15165222.0,0,0.0,198.0,0.0,0.0
16,165,1,249,5596619.0,190,170323000.0,96.0,73.0,72721000.0
47,504,1,410,12087624.0,217,128469500.0,251.0,140.0,83164000.0
...,...,...,...,...,...,...,...,...,...
2185673,17171661,1,831,12403445.0,591,650315000.0,428.0,391.0,386392000.0
2185695,17171710,1,100,16980619.0,81,137863000.0,53.0,43.0,80225000.0
2185944,17172214,1,235,7277282.0,152,43243500.0,119.0,76.0,14953500.0
2186182,17172815,1,288,10476722.0,268,172972000.0,188.0,176.0,116695000.0


In [19]:
fact_data_year_pass_thu_ho = fact_data_year_pass[(fact_data_year_pass['thu_ho'] != 0) | (fact_data_year_pass['thuho_tongdon'] != 0)].reset_index(drop = True)
fact_data_year_pass_thu_ho

,cus_id,tgsd_flag,tong_don,tong_tien,tongdon_cod,thuho_tongdon,don_ptc,don_ptc_cod,thu_ho
0,20,1,764,16775286.0,297,219789000.0,627.0,248.0,174743000.0
1,78,1,121,4872605.0,22,13459000.0,34.0,8.0,5417000.0
2,165,1,249,5596619.0,190,170323000.0,96.0,73.0,72721000.0
3,504,1,410,12087624.0,217,128469500.0,251.0,140.0,83164000.0
4,1092,1,759,29203098.0,547,780047000.0,612.0,443.0,614159000.0
...,...,...,...,...,...,...,...,...,...
45563,17171661,1,831,12403445.0,591,650315000.0,428.0,391.0,386392000.0
45564,17171710,1,100,16980619.0,81,137863000.0,53.0,43.0,80225000.0
45565,17172214,1,235,7277282.0,152,43243500.0,119.0,76.0,14953500.0
45566,17172815,1,288,10476722.0,268,172972000.0,188.0,176.0,116695000.0


In [27]:
fact_data_year_pass_thu_ho['rev_per_don_ptc'] =  (fact_data_year_pass_thu_ho['thu_ho'] / fact_data_year_pass_thu_ho['don_ptc_cod']).fillna(0)
fact_data_year_pass_thu_ho['rev_per_tongdon'] =  (fact_data_year_pass_thu_ho['thuho_tongdon'] / fact_data_year_pass_thu_ho['tongdon_cod']).fillna(0)

In [35]:
# tính trung bình 
fact_data_year_pass_thu_ho['gmv_case1'] = fact_data_year_pass_thu_ho['don_ptc'] * ( ( fact_data_year_pass_thu_ho['rev_per_don_ptc'] + fact_data_year_pass_thu_ho['rev_per_tongdon'] ) / 2)

In [36]:
# tính theo ưu tiên đơn ptc có cod
fact_data_year_pass_thu_ho['gmv_case2'] = fact_data_year_pass_thu_ho['don_ptc'] * (fact_data_year_pass_thu_ho[
    'rev_per_don_ptc'
].fillna(fact_data_year_pass_thu_ho['rev_per_tongdon']))

In [25]:
fact_data_year_pass_thu_ho['rev_per_tongdon'].describe()

count    7.430000e+04
mean     1.213199e+06
std      1.778163e+06
min      1.000000e+00
25%      3.805513e+05
50%      7.284952e+05
75%      1.427096e+06
max      1.300000e+08
Name: rev_per_tongdon, dtype: float64

In [37]:
fact_data_year_pass_thu_ho['gmv_case1'].describe(
    percentiles=[i/100 for i in range(5, 100, 5)]
)

count    7.431400e+04
mean     1.363782e+09
std      2.523347e+10
min      6.150000e+01
5%       2.009259e+07
10%      3.628030e+07
15%      5.337394e+07
20%      7.285056e+07
25%      9.602109e+07
30%      1.226426e+08
35%      1.540104e+08
40%      1.928182e+08
45%      2.383884e+08
50%      2.957150e+08
55%      3.646191e+08
60%      4.534422e+08
65%      5.688400e+08
70%      7.168498e+08
75%      9.181566e+08
80%      1.212186e+09
85%      1.673872e+09
90%      2.468785e+09
95%      4.384164e+09
max      6.062528e+12
Name: gmv_case1, dtype: float64

In [38]:
2.009259e+07

20092590.0

In [39]:
fact_data_year_pass_thu_ho['gmv_case2'].describe(
    percentiles=[i/100 for i in range(5, 100, 5)]
)

count    7.431400e+04
mean     1.349943e+09
std      2.482317e+10
min      0.000000e+00
5%       1.342264e+07
10%      3.080922e+07
15%      4.836029e+07
20%      6.788696e+07
25%      9.111259e+07
30%      1.174704e+08
35%      1.491984e+08
40%      1.870596e+08
45%      2.334574e+08
50%      2.908930e+08
55%      3.590080e+08
60%      4.478117e+08
65%      5.629465e+08
70%      7.127473e+08
75%      9.126088e+08
80%      1.201823e+09
85%      1.662048e+09
90%      2.453094e+09
95%      4.354215e+09
max      5.950514e+12
Name: gmv_case2, dtype: float64

In [41]:
fact_data_year_pass_thu_ho['chi_phi_vtp'] = fact_data_year_pass_thu_ho['tong_tien'] / fact_data_year_pass_thu_ho['gmv_case1']

In [43]:
fact_data_year_pass_thu_ho[fact_data_year_pass_thu_ho['tong_tien'] <= 1000000]['chi_phi_vtp'].describe(
    percentiles=[i/100 for i in range(5, 100, 5)]
)

count    19.000000
mean      0.702966
std       0.586091
min       0.063178
5%        0.091794
10%       0.117982
15%       0.186362
20%       0.239415
25%       0.261677
30%       0.288247
35%       0.331214
40%       0.359951
45%       0.382177
50%       0.412330
55%       0.542535
60%       0.715945
65%       0.813483
70%       0.968949
75%       1.072946
80%       1.206814
85%       1.442984
90%       1.618149
95%       1.762732
max       1.854351
Name: chi_phi_vtp, dtype: float64

In [44]:
fact_data_year_pass_thu_ho[fact_data_year_pass_thu_ho['chi_phi_vtp'] > 1]

,cus_id,tong_don,tong_tien,tongdon_cod,thuho_tongdon,don_ptc,don_ptc_cod,thu_ho,rev_per_don_ptc,rev_per_tongdon,gmv_case1,gmv_case2,chi_phi_vtp
26,3275,2507,291729634.0,2,309000.0,2332.0,1.0,50000.0,50000.000000,154500.000000,2.384470e+08,1.166000e+08,1.223457
49,6968,199,6716707.0,117,6782000.0,93.0,61.0,4812000.0,78885.245902,57965.811966,6.363574e+06,7.336328e+06,1.055493
65,11475,2287,31749234.0,829,7282901.0,2116.0,784.0,6923100.0,8830.484694,8785.164053,1.863736e+07,1.868531e+07,1.703527
83,18368,297,24123584.0,2,155000.0,147.0,1.0,120000.0,120000.000000,77500.000000,1.451625e+07,1.764000e+07,1.661833
185,35763,2788,145305481.0,1,35000.0,2363.0,1.0,35000.0,35000.000000,35000.000000,8.270500e+07,8.270500e+07,1.756913
...,...,...,...,...,...,...,...,...,...,...,...,...,...
74000,17104290,536,14252700.0,488,3509500.0,288.0,271.0,2160000.0,7970.479705,7191.598361,2.183339e+06,2.295498e+06,6.527937
74054,17115958,197,5472878.0,4,100000.0,51.0,2.0,52000.0,26000.000000,25000.000000,1.300500e+06,1.326000e+06,4.208288
74076,17120141,509,22068149.0,338,14263000.0,254.0,151.0,1495000.0,9900.662252,42198.224852,6.616559e+06,2.514768e+06,3.335291
74128,17127382,86,3564465.0,23,785000.0,34.0,11.0,340000.0,30909.090909,34130.434783,1.105672e+06,1.050909e+06,3.223800


In [20]:
fact_data_year_pass_not_thu_ho = fact_data_year_pass[(fact_data_year_pass['thu_ho'] == 0) & (fact_data_year_pass['thuho_tongdon'] == 0)].reset_index(drop = True)
fact_data_year_pass_not_thu_ho

,cus_id,tgsd_flag,tong_don,tong_tien,tongdon_cod,thuho_tongdon,don_ptc,don_ptc_cod,thu_ho
0,112,1,313,15165222.0,0,0.0,198.0,0.0,0.0
1,701,1,141,5055207.0,0,0.0,58.0,0.0,0.0
2,709,1,486,7334600.0,0,0.0,282.0,0.0,0.0
3,1173,1,561,15911581.0,0,0.0,396.0,0.0,0.0
4,1185,1,439,14768699.0,0,0.0,321.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...
27730,17159275,1,275,6066381.0,0,0.0,160.0,0.0,0.0
27731,17162019,1,57,3100360.0,0,0.0,25.0,0.0,0.0
27732,17163883,1,524,38607969.0,0,0.0,473.0,0.0,0.0
27733,17167553,1,61,3126437.0,0,0.0,27.0,0.0,0.0


In [21]:
45568 + 27735 

73303

In [10]:
fact_data_year_pass_not_thu_ho[fact_data_year_pass_not_thu_ho['tong_tien'] == 0]

,cus_id,tong_don,tong_tien,tongdon_cod,thuho_tongdon,don_ptc,don_ptc_cod,thu_ho
2314,1937971,235,0.0,0,0.0,112.0,0.0,0.0
30063,15432565,181,0.0,0,0.0,107.0,0.0,0.0


In [11]:
fact_data_year_pass_not_thu_ho['chi_phi_per_don'] = fact_data_year_pass_not_thu_ho['tong_tien'] / fact_data_year_pass_not_thu_ho['tong_don']

In [32]:
1.749119e+04

17491.19

In [12]:
fact_data_year_pass_not_thu_ho['chi_phi_per_don'].describe(
    percentiles=[i/100 for i in range(5, 100, 5)]
)

count    3.401000e+04
mean     5.144087e+04
std      3.356614e+05
min      0.000000e+00
5%       1.272966e+04
10%      1.385088e+04
15%      1.490267e+04
20%      1.612082e+04
25%      1.749119e+04
30%      1.887115e+04
35%      2.028209e+04
40%      2.175654e+04
45%      2.327845e+04
50%      2.495400e+04
55%      2.678349e+04
60%      2.884329e+04
65%      3.122306e+04
70%      3.431064e+04
75%      3.845160e+04
80%      4.426345e+04
85%      5.306011e+04
90%      6.766482e+04
95%      1.045089e+05
max      2.684789e+07
Name: chi_phi_per_don, dtype: float64

In [37]:
1.272966e+04

12729.66

In [16]:
fact_data_year_pass_not_thu_ho['est_gmv'] = (fact_data_year_pass_not_thu_ho['don_ptc'] * fact_data_year_pass_not_thu_ho['chi_phi_per_don'] ) / 0.2
fact_data_year_pass_not_thu_ho['est_gmv'].describe(
    percentiles=[i/100 for i in range(5, 100, 5)]
)

count    3.401000e+04
mean     1.124762e+08
std      3.639838e+09
min      0.000000e+00
5%       2.755991e+06
10%      3.768977e+06
15%      4.803708e+06
20%      5.874015e+06
25%      6.997536e+06
30%      8.273890e+06
35%      9.705809e+06
40%      1.141628e+07
45%      1.355906e+07
50%      1.608649e+07
55%      1.922249e+07
60%      2.321545e+07
65%      2.803322e+07
70%      3.442963e+07
75%      4.342620e+07
80%      5.710887e+07
85%      7.941030e+07
90%      1.202205e+08
95%      2.363203e+08
max      6.245162e+11
Name: est_gmv, dtype: float64

In [17]:
1.608649e+07

16086490.0

In [35]:
fact_data_year_pass_not_thu_ho['percentile_5'].value_counts()

percentile_5
19    1701
4     1701
13    1701
11    1701
0     1701
15    1701
8     1701
17    1701
6     1701
2     1701
5     1700
16    1700
3     1700
14    1700
18    1700
12    1700
10    1700
1     1700
9     1700
7     1700
Name: count, dtype: int64

In [ ]:
fact_data_year_notpass

In [28]:
df_temp = pd.DataFrame(list_cus_trong, columns = ['cus_id'])
df_temp

,cus_id
0,1
1,7
2,28
3,39
4,63
...,...
1644749,18158971
1644750,18159038
1644751,18159060
1644752,18159087


In [21]:
rule_dataset3 = rule_dataset2[rule_dataset2['cus_id'].isin(list_cus_trong)]
rule_dataset3.shape

SyntaxError: invalid syntax (3162195979.py, line 1)

In [31]:
list_cus_trong_set = set(list_cus_trong)

rule_dataset3 = rule_dataset2.loc[
    ~rule_dataset2['cus_id'].isin(list_cus_trong_set)
]
rule_dataset3.shape

(854827, 3)

In [25]:
print(2189209-1644754)

544455


In [30]:
# Sử dụng dấu ~ để phủ định điều kiện isin
fact_data_3 = fact_data[fact_data['cus_id'].isin(list_cust)]
fact_data_3

,index,cus_id,tong_don,tongdon_cod,thuho_tongdon,don_ptc,don_ptc_cod,thu_ho,month
23,23,10000725,29,0,0.0,13.0,0.0,0.0,202504
32,32,10001064,240,176,131930000.0,218.0,164.0,130380000.0,202504
40,40,10001236,482,410,418589000.0,462.0,392.0,394408000.0,202504
67,67,10002116,28,17,6930000.0,19.0,11.0,4760000.0,202504
68,68,10002126,5,0,0.0,1.0,0.0,0.0,202504
...,...,...,...,...,...,...,...,...,...
9039609,782662,9998707,16,11,5735000.0,4.0,3.0,830000.0,202603
9039622,782675,9999008,77,18,4079000.0,59.0,20.0,4617000.0,202603
9039624,782677,9999040,20,9,6320000.0,10.0,3.0,700000.0,202603
9039641,782694,9999337,47,7,4530000.0,26.0,6.0,4080000.0,202603


In [32]:
print("Kiểu dữ liệu trong bảng fact:", fact_data['cus_id'].dtype)
print("Kiểu dữ liệu trong danh sách list_cus:", type(list_cus[0]))

Kiểu dữ liệu trong bảng fact: int64
Kiểu dữ liệu trong danh sách list_cus: <class 'int'>


In [33]:
set_fact = set(fact_data['cus_id'])
set_list = set(list_cus)

giao_thoa = set_fact.intersection(set_list)
chi_co_trong_fact = set_fact - set_list

print(f"Số ID thực tế có trong cả hai (bị loại bỏ): {len(giao_thoa)}")
print(f"Số ID đáng lẽ phải còn lại: {len(chi_co_trong_fact)}")

Số ID thực tế có trong cả hai (bị loại bỏ): 1644754
Số ID đáng lẽ phải còn lại: 877868


In [31]:
print(len(fact_data_3['cus_id'].unique()))

185255


In [ ]:
fact_data_3['']

In [14]:
fact_data_12_month = fact_data.groupby("cus_id").agg({
            "tong_don": "sum",
            "tongdon_cod": "sum",
            "thuho_tongdon": "sum",
            "don_ptc": "sum",
            "don_ptc_cod": "sum",
            "thu_ho": "sum"
        }).reset_index()
fact_data_12_month

,cus_id,tong_don,tongdon_cod,thuho_tongdon,don_ptc,don_ptc_cod,thu_ho
0,1,21,8,4634000.0,2.0,1.0,220000.0
1,7,189,118,182412000.0,62.0,37.0,49362000.0
2,20,764,297,219789000.0,627.0,248.0,174743000.0
3,28,1,0,0.0,0.0,0.0,0.0
4,34,5,1,6400000.0,1.0,0.0,0.0
...,...,...,...,...,...,...,...
2522617,18158971,1,0,0.0,0.0,0.0,0.0
2522618,18159038,1,0,0.0,0.0,0.0,0.0
2522619,18159060,2,0,0.0,0.0,0.0,0.0
2522620,18159087,2,0,0.0,0.0,0.0,0.0


fact_data.shape

In [39]:
fact_data = pd.merge(dim_data_flag[['cus_id']], fact_data, how = 'inner', on = 'cus_id')
fact_data

,cus_id,tong_don,tong_tien,don_ptc,thu_ho,don_hoan,month
0,12302021,1190,17629250.0,1129.0,0.0,3.0,202503
1,12302021,229,3839643.0,224.0,0.0,2.0,202510
2,12302021,360,5316178.0,373.0,0.0,2.0,202508
3,12302021,549,7715974.0,556.0,0.0,1.0,202506
4,12302021,851,12323782.0,859.0,0.0,1.0,202504
...,...,...,...,...,...,...,...
10384150,2872143,1,79001.0,0.0,0.0,0.0,202601
10384151,12059998,1,33501.0,0.0,0.0,0.0,202601
10384152,149097,1,19999.0,0.0,0.0,0.0,202601
10384153,17368379,1,107604.0,0.0,0.0,0.0,202601


In [49]:
fact_data['avg_tien_don'] = fact_data['tong_tien'] / fact_data['tong_don']

In [50]:
fact_data['avg_tien_don'].describe()

count    1.038416e+07
mean     6.615671e+04
std      2.595449e+06
min     -1.700000e+04
25%      2.350000e+04
50%      3.184573e+04
75%      4.689967e+04
max      5.282159e+09
Name: avg_tien_don, dtype: float64

In [52]:
4.689967e+04

46899.67

In [40]:
pivot_fact_data = fact_data.pivot_table(
    index='cus_id',
    columns='month',
    values='tong_don',
    aggfunc='sum'
)
pivot_fact_data = pivot_fact_data.sort_index(axis=1)
pivot_fact_data

month,202503,202504,202505,202506,202507,202508,202509,202510,202511,202512,202601,202602
cus_id,,,,,,,,,,,,
1,1.0,2.0,7.0,3.0,2.0,1.0,2.0,1.0,1.0,1.0,1.0,NaN
7,34.0,54.0,38.0,52.0,44.0,34.0,40.0,30.0,24.0,20.0,18.0,14.0
20,132.0,144.0,162.0,112.0,140.0,88.0,134.0,96.0,142.0,124.0,140.0,132.0
28,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,NaN
34,2.0,1.0,1.0,1.0,1.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
17590726,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,NaN
17590739,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,8.0,NaN
17590740,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,NaN


In [41]:

# tạo mask True nếu là NaN
mask = pivot_fact_data.isna()

# kiểm tra 3 tháng liên tiếp đều NaN
result3 = mask.astype(int).rolling(window=3, axis=1).sum() == 3
result6 = mask.astype(int).rolling(window=6, axis=1).sum() == 6

pivot_fact_data['is_3_months_0_don'] = result3.any(axis=1)
pivot_fact_data['is_6_months_0_don'] = result6.any(axis=1)
pivot_fact_data

/tmp/ipykernel_1967241/1550725288.py:5: FutureWarning: Support for axis=1 in DataFrame.rolling is deprecated and will be removed in a future version. Use obj.T.rolling(...) instead
  result3 = mask.astype(int).rolling(window=3, axis=1).sum() == 3
/tmp/ipykernel_1967241/1550725288.py:6: FutureWarning: Support for axis=1 in DataFrame.rolling is deprecated and will be removed in a future version. Use obj.T.rolling(...) instead
  result6 = mask.astype(int).rolling(window=6, axis=1).sum() == 6


month,202503,202504,202505,202506,202507,202508,202509,202510,202511,202512,202601,202602,is_3_months_0_don,is_6_months_0_don
cus_id,,,,,,,,,,,,,,
1,1.0,2.0,7.0,3.0,2.0,1.0,2.0,1.0,1.0,1.0,1.0,NaN,False,False
7,34.0,54.0,38.0,52.0,44.0,34.0,40.0,30.0,24.0,20.0,18.0,14.0,False,False
20,132.0,144.0,162.0,112.0,140.0,88.0,134.0,96.0,142.0,124.0,140.0,132.0,False,False
28,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,NaN,True,True
34,2.0,1.0,1.0,1.0,1.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,True,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17590726,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,NaN,True,True
17590739,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,8.0,NaN,True,True
17590740,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,NaN,True,True


In [42]:
summary = pd.DataFrame({
    'mean_orders': pivot_fact_data.select_dtypes(include='number').fillna(0).mean(axis=1),
    'median_orders': pivot_fact_data.select_dtypes(include='number').fillna(0).median(axis=1),
    'active_months': pivot_fact_data.select_dtypes(include='number').notna().sum(axis=1)
})
summary

,mean_orders,median_orders,active_months
cus_id,,,
1,1.833333,1.0,11
7,33.500000,34.0,12
20,128.833333,133.0,12
28,0.083333,0.0,1
34,0.583333,0.5,6
...,...,...,...
17590726,0.083333,0.0,1
17590739,0.666667,0.0,1
17590740,0.083333,0.0,1


#### Tong_don

In [43]:
pivot_fact_data = pivot_fact_data.reset_index()
tong_don = pd.merge(summary,pivot_fact_data[['cus_id','is_3_months_0_don','is_6_months_0_don']], how = 'inner', on = 'cus_id' )
tong_don['is_3_months_0_don'] = tong_don['is_3_months_0_don'].astype(int)
tong_don['is_6_months_0_don'] = tong_don['is_6_months_0_don'].astype(int)
tong_don

,cus_id,mean_orders,median_orders,active_months,is_3_months_0_don,is_6_months_0_don
0,1,1.833333,1.0,11,0,0
1,7,33.500000,34.0,12,0,0
2,20,128.833333,133.0,12,0,0
3,28,0.083333,0.0,1,1,1
4,34,0.583333,0.5,6,1,1
...,...,...,...,...,...,...
2390330,17590726,0.083333,0.0,1,1,1
2390331,17590739,0.666667,0.0,1,1,1
2390332,17590740,0.083333,0.0,1,1,1
2390333,17590745,0.083333,0.0,1,1,1


In [44]:
tong_don['is_3_months_0_don'].value_counts()

is_3_months_0_don
1    1972705
0     417630
Name: count, dtype: int64

In [9]:
tong_don['is_6_months_0_don'].value_counts()

is_6_months_0_don
1    1596041
0     897918
Name: count, dtype: int64

In [45]:
tong_don[tong_don['is_3_months_0_don'] == 0]['active_months'].value_counts().sort_index()

active_months
4       2175
5       8403
6      17044
7      25507
8      33461
9      40424
10     53205
11     61258
12    176153
Name: count, dtype: int64

In [46]:
fillter_data = tong_don[(tong_don['is_3_months_0_don'] == 0) & (tong_don['active_months'] >= 6)]
fillter_data

,cus_id,mean_orders,median_orders,active_months,is_3_months_0_don,is_6_months_0_don
0,1,1.833333,1.0,11,0,0
1,7,33.500000,34.0,12,0,0
2,20,128.833333,133.0,12,0,0
8,78,20.500000,18.0,12,0,0
10,112,29.750000,20.0,12,0,0
...,...,...,...,...,...,...
1922464,16266960,5.666667,6.0,10,0,0
1922489,16267030,5.083333,6.0,10,0,0
1922502,16267052,1.000000,1.0,7,0,0
1922518,16267112,2.250000,0.5,6,0,0


In [12]:
fillter_data['mean_orders'].describe()

count    4.074370e+05
mean     4.708267e+01
std      4.036232e+03
min      5.000000e-01
25%      2.000000e+00
50%      5.166667e+00
75%      1.758333e+01
max      2.562575e+06
Name: mean_orders, dtype: float64

In [16]:
2.000000e+00

2.0

In [17]:
5.166667e+00

5.166667

In [19]:
1.758333e+01

17.58333

In [47]:
fillter_data[fillter_data['mean_orders'] * fillter_data['active_months'] >= 100]

,cus_id,mean_orders,median_orders,active_months,is_3_months_0_don,is_6_months_0_don
1,7,33.500000,34.0,12,0,0
2,20,128.833333,133.0,12,0,0
8,78,20.500000,18.0,12,0,0
10,112,29.750000,20.0,12,0,0
12,125,67.916667,71.0,12,0,0
...,...,...,...,...,...,...
1920398,16261541,85.833333,78.0,10,0,0
1920426,16261608,76.916667,83.5,10,0,0
1922277,16266475,374.000000,423.5,10,0,0
1922299,16266524,25.750000,21.0,9,0,0


In [15]:
fillter_data['mean_orders_quintile'] = pd.qcut(
    tong_don['mean_orders'],
    q=20,           # 20 nhóm = quintile
    labels=False,  # nhãn 0..4
    duplicates='drop'  # bỏ các cạnh trùng nhau nếu có
)

fillter_data

/tmp/ipykernel_1967241/3705183934.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  fillter_data['mean_orders_quintile'] = pd.qcut(


,cus_id,mean_orders,median_orders,active_months,is_3_months_0_don,is_6_months_0_don,mean_orders_quintile
0,1,1.833333,1.0,11,0,0,7
1,7,16.750000,17.0,12,0,0,9
2,20,64.416667,66.5,12,0,0,10
8,78,10.250000,9.0,12,0,0,9
10,112,29.750000,20.0,12,0,0,10
...,...,...,...,...,...,...,...
1923155,16266960,5.666667,6.0,10,0,0,8
1923180,16267030,5.083333,6.0,10,0,0,8
1923193,16267052,1.000000,1.0,7,0,0,5
1923209,16267112,2.250000,0.5,6,0,0,7


In [70]:
fillter_data['mean_orders_quintile'].value_counts()

mean_orders_quintile
10    89144
11    84240
9     80410
8     62275
7     42565
6     20556
5      9055
4       897
Name: count, dtype: int64

In [34]:
pivot_fact_data['is_3_months_0_don'] = result.any(axis=1)
pivot_fact_data

month,202502,202503,202504,202505,202506,202507,202508,202509,202510,202511,202512,202601,202602,is_3_months_0_don
cus_id,,,,,,,,,,,,,,
1,NaN,1.0,2.0,7.0,3.0,2.0,1.0,2.0,1.0,1.0,1.0,1.0,NaN,False
7,20.0,17.0,27.0,19.0,26.0,22.0,17.0,20.0,15.0,12.0,10.0,9.0,7.0,False
20,65.0,66.0,72.0,81.0,56.0,70.0,44.0,67.0,48.0,71.0,62.0,70.0,66.0,False
28,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,NaN,True
34,2.0,2.0,1.0,1.0,1.0,1.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17958144,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,True
17958204,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,True
17958248,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,True


In [35]:
pivot_fact_data['is_3_months_0_don'].value_counts()

is_3_months_0_don
True     2197954
False     393252
Name: count, dtype: int64

In [27]:
result = mask.astype(int).rolling(window=3, axis=1).sum() == 3
result

/tmp/ipykernel_1961474/2895308849.py:1: FutureWarning: Support for axis=1 in DataFrame.rolling is deprecated and will be removed in a future version. Use obj.T.rolling(...) instead
  result = mask.astype(int).rolling(window=3, axis=1).sum() == 3


month,1970-01-01 00:00:00.000202502,1970-01-01 00:00:00.000202503,1970-01-01 00:00:00.000202504,1970-01-01 00:00:00.000202505,1970-01-01 00:00:00.000202506,1970-01-01 00:00:00.000202507,1970-01-01 00:00:00.000202508,1970-01-01 00:00:00.000202509,1970-01-01 00:00:00.000202510,1970-01-01 00:00:00.000202511,1970-01-01 00:00:00.000202512,1970-01-01 00:00:00.000202601,1970-01-01 00:00:00.000202602
cus_id,,,,,,,,,,,,,
1,False,False,False,False,False,False,False,False,False,False,False,False,False
7,False,False,False,False,False,False,False,False,False,False,False,False,False
20,False,False,False,False,False,False,False,False,False,False,False,False,False
28,False,False,True,True,True,True,True,True,True,True,True,False,False
34,False,False,False,False,False,False,False,False,False,True,True,True,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...
17958144,False,False,True,True,True,True,True,True,True,True,True,True,False
17958204,False,False,True,True,True,True,True,True,True,True,True,True,False
17958248,False,False,True,True,True,True,True,True,True,True,True,True,False


In [28]:
customers_3_nan = result.any(axis=1)
customers_3_nan

cus_id
1           False
7           False
20          False
28           True
34           True
            ...  
17958144     True
17958204     True
17958248     True
17958468     True
17958505     True
Length: 2591206, dtype: bool

In [17]:
# df_pivot: index = cus_id, columns = tháng

# tạo mask True nếu là NaN
mask = pivot_fact_data.isna()

# kiểm tra 3 tháng liên tiếp đều NaN
result = mask.rolling(window=3, axis=1).sum() == 3

# khách hàng có ít nhất 1 đoạn 3 tháng liên tiếp NaN
customers_3_nan = result.any(axis=1)

# lấy danh sách cus_id
list_cus = pivot_fact_data.index[customers_3_nan]
list_cus

/tmp/ipykernel_1961474/3931850300.py:7: FutureWarning: Support for axis=1 in DataFrame.rolling is deprecated and will be removed in a future version. Use obj.T.rolling(...) instead
  result = mask.rolling(window=3, axis=1).sum() == 3


DataError: No numeric types to aggregate

#### Tính cho dim_data

In [21]:
dim_data = pd.read_csv(r'../monthly_grouped_data/dim_data/12month_dim_data.csv')
dim_data.drop(columns=['index'], inplace = True)
dim_data.head()     

,cus_id,ngay_hoptac,ngay_sinh,ten_dangky,nganh_hang,loai_khachhang
0,12302021,27/11/2022,NaN,BÙI THỊ KIM NGÂN,Khác,Hộ kinh doanh cá thể
1,14724335,08/07/2024,NaN,PHẠM NGỌC RIN,Bán lẻ,Cá nhân
2,15720941,08/02/2025,NaN,PHẠM NHƯ Ý,Khác,Cá nhân
3,14027680,12/03/2024,NaN,NGUYỄN VĂN ĐĂNG,Khác,Hộ kinh doanh cá thể
4,11290277,19/03/2022,NaN,NGUYỄN THỊ HẰNG,Khác,Cá nhân


In [22]:
dim_data.shape

(2659413, 6)

In [23]:
import datetime as datetime
import numpy as np
dim_data['ngay_hoptac'] = pd.to_datetime(dim_data['ngay_hoptac'], format='%d/%m/%Y')
dim_data['thoigian_sd'] = (pd.Timestamp.now().year - dim_data['ngay_hoptac'].dt.year)*12 - (pd.Timestamp.now().month - dim_data['ngay_hoptac'].dt.month)
dim_data['tgsd_flag'] = np.where(dim_data['thoigian_sd'] >= 6, 1,0)
dim_data.head()

,cus_id,ngay_hoptac,ngay_sinh,ten_dangky,nganh_hang,loai_khachhang,thoigian_sd,tgsd_flag
0,12302021,2022-11-27,NaN,BÙI THỊ KIM NGÂN,Khác,Hộ kinh doanh cá thể,55.0,1
1,14724335,2024-07-08,NaN,PHẠM NGỌC RIN,Bán lẻ,Cá nhân,27.0,1
2,15720941,2025-02-08,NaN,PHẠM NHƯ Ý,Khác,Cá nhân,10.0,1
3,14027680,2024-03-12,NaN,NGUYỄN VĂN ĐĂNG,Khác,Hộ kinh doanh cá thể,23.0,1
4,11290277,2022-03-19,NaN,NGUYỄN THỊ HẰNG,Khác,Cá nhân,47.0,1


In [25]:
dim_data_flag = dim_data[dim_data['tgsd_flag'] == 1]
dim_data_flag

,cus_id,ngay_hoptac,ngay_sinh,ten_dangky,nganh_hang,loai_khachhang,thoigian_sd,tgsd_flag
0,12302021,2022-11-27,NaN,BÙI THỊ KIM NGÂN,Khác,Hộ kinh doanh cá thể,55.0,1
1,14724335,2024-07-08,NaN,PHẠM NGỌC RIN,Bán lẻ,Cá nhân,27.0,1
2,15720941,2025-02-08,NaN,PHẠM NHƯ Ý,Khác,Cá nhân,10.0,1
3,14027680,2024-03-12,NaN,NGUYỄN VĂN ĐĂNG,Khác,Hộ kinh doanh cá thể,23.0,1
4,11290277,2022-03-19,NaN,NGUYỄN THỊ HẰNG,Khác,Cá nhân,47.0,1
...,...,...,...,...,...,...,...,...
2659398,2872143,2018-12-21,NaN,Hạnh Nhi,Khác,Cá nhân,104.0,1
2659402,12059998,2022-09-15,NaN,NaN,NaN,NaN,53.0,1
2659405,149097,2017-09-28,NaN,Công ty CPĐT Thăng Long Xanh ĐÃ CHUYỂN LVL98,Khác,Doanh nghiệp vừa và nhỏ (SME),113.0,1
2659406,17368379,2025-11-27,NaN,NaN,NaN,NaN,19.0,1


In [10]:
dim_data['tgsd_flag'].value_counts()

tgsd_flag
1    2653498
0     104199
Name: count, dtype: int64

In [12]:
dim_data['loai_khachhang'].value_counts()

loai_khachhang
Cá nhân                          1828852
Hộ kinh doanh cá thể              183444
Doanh nghiệp vừa và nhỏ (SME)      67726
Doanh nghiệp lớn                    7695
Khách hàng chính phủ                 633
Sàn                                  211
Cổng vận chuyển                      194
Name: count, dtype: int64

#### Tính cho fact_data

In [ ]:
fact_data[fact_data['cus_id'] == 10000085].to_csv(r'../monthly_grouped_data/Untitled Folder/cusid_10000085.csv', index = False)